## Código ordenado: modelo de regresión lineal con comuna, hora, día, cantidad de paquetes, etc.

In [ ]:
#Librerías
import pandas as pd
import numpy as np

In [ ]:
# Los datos cargados en este bloque consideran dos empresas: Falabella (32849) y CoAndina (80058)

# Datos gps son todos los puntos lat,lon de trayectoria gps de vehículos entre el 22 y 29 de septiembre de 2025
gps = pd.read_csv("gps.csv")
# Datos visitas son todos los puntos lat,lon de paradas programadas + lat,lon checkouts 
visitas = pd.read_csv("visitas2.csv")

In [ ]:
visitas.info()

In [ ]:
# Separamos los datos por empresa tanto para gps como para visitas
gps_falabella = gps[gps["account_id"]== 32849].copy()
gps_CoAndina = gps[gps["account_id"]== 80058].copy()

visitas_falabella = visitas[visitas["account_id"]== 32849].copy()
visitas_CoAndina = visitas[visitas["account_id"]== 80058].copy()

In [ ]:
## Debemos agregar variable día de la semana, para poder modificar el código en adelante, a traves de la columna DIA
import pandas as pd

# Asegurar que created sea datetime con zona horaria
gps_falabella["created"] = pd.to_datetime(gps_falabella["created"], utc=True)

# Crear columna con nombre del día en español
dias = {
    "Monday": "lunes",
    "Tuesday": "martes",
    "Wednesday": "miércoles",
    "Thursday": "jueves",
    "Friday": "viernes",
    "Saturday": "sábado",
    "Sunday": "domingo"
}

gps_falabella["dia"] = gps_falabella["created"].dt.day_name().map(dias)
gps_22 = gps_falabella

In [ ]:
import pandas as pd

visitas_falabella["event_date"] = pd.to_datetime(
    visitas_falabella["event_date"],
    format="mixed",
    utc=True
)

# Diccionario días en español

dias = {
    "Monday": "lunes",
    "Tuesday": "martes",
    "Wednesday": "miércoles",
    "Thursday": "jueves",
    "Friday": "viernes",
    "Saturday": "sábado",
    "Sunday": "domingo"
}

# Crear columna día (en UTC)
visitas_falabella["dia"] = visitas_falabella["event_date"].dt.day_name().map(dias)
visitas_F22 = visitas_falabella

In [ ]:
THRESHOLD = 0.1
MAX_MINUTES = 5

gps_22["created"] = pd.to_datetime(gps_22["created"], errors="coerce", utc=True)
gps_22["battery_level"] = pd.to_numeric(gps_22["battery_level"], errors="coerce")

gps_22 = gps_22.dropna(subset=["owner_id", "created", "battery_level"])

# Día en hora Chile
dias = {
    "Monday": "lunes",
    "Tuesday": "martes",
    "Wednesday": "miércoles",
    "Thursday": "jueves",
    "Friday": "viernes",
    "Saturday": "sábado",
    "Sunday": "domingo"
}

gps_22["created_cl"] = gps_22["created"].dt.tz_convert("America/Santiago")
gps_22["dia"] = gps_22["created_cl"].dt.day_name().map(dias)

# Orden correcto
gps_22 = gps_22.sort_values(["owner_id", "dia", "created_cl"])

# Total de rutas reales de la semana
rutas_totales = (
    gps_22[["owner_id", "dia"]]
    .drop_duplicates()
)

total_rutas_semana = len(rutas_totales)

# Diferencias dentro de cada ruta owner_id-día
gps_22["battery_jump"] = (
    gps_22.groupby(["owner_id", "dia"])["battery_level"]
    .diff()
)

gps_22["time_diff_min"] = (
    gps_22.groupby(["owner_id", "dia"])["created_cl"]
    .diff()
    .dt.total_seconds() / 60
)

# Ruta sospechosa si dentro de ese owner_id-día hay salto raro
gps_22["is_suspicious_jump"] = (
    (gps_22["battery_jump"].abs() > THRESHOLD) &
    (gps_22["time_diff_min"] < MAX_MINUTES)
)

# Rutas sospechosas owner_id-día
rutas_sospechosas = (
    gps_22.loc[gps_22["is_suspicious_jump"], ["owner_id", "dia"]]
    .drop_duplicates()
)

total_rutas_sospechosas = len(rutas_sospechosas)

# Owner_id que fueron sospechosos más de un día
owner_sospechosos_mas_de_un_dia = (
    rutas_sospechosas
    .groupby("owner_id")["dia"]
    .nunique()
    .reset_index(name="dias_sospechosos")
    .query("dias_sospechosos > 1")
)

# Eliminar SOLO las rutas owner_id-día sospechosas
gps_22_limpio = (
    gps_22
    .merge(
        rutas_sospechosas.assign(ruta_sospechosa=1),
        on=["owner_id", "dia"],
        how="left"
    )
)

gps_22_limpio = gps_22_limpio[gps_22_limpio["ruta_sospechosa"].isna()].copy()
gps_22_limpio = gps_22_limpio.drop(columns=["ruta_sospechosa"])

# Rutas finales limpias
rutas_limpias = (
    gps_22_limpio[["owner_id", "dia"]]
    .drop_duplicates()
)

print(f"Rutas totales semana: {total_rutas_semana}")
print(f"Rutas sospechosas eliminadas: {total_rutas_sospechosas}")
print(f"Rutas limpias finales: {len(rutas_limpias)}")
print(f"Owner_id sospechosos más de un día: {len(owner_sospechosos_mas_de_un_dia)}")

In [ ]:
# =========================================================
# AGRUPAR CHECKOUTS EN "PARADAS REALIZADAS"
# Criterio:
# - mismo user_id
# - mismo día
# - ordenados por event_date
# - misma parada si siguen dentro de un radio de 10 m
#   respecto al ancla de la parada actual
# =========================================================

import pandas as pd
import numpy as np

RANGO_PARADA_M = 10

# Normalizar tipos por seguridad
visitas_F22["user_id"] = pd.to_numeric(visitas_F22["user_id"], errors="coerce")
visitas_F22["event_date"] = pd.to_datetime(
    visitas_F22["event_date"],
    errors="coerce",
    utc=True
)
visitas_F22["latitude_1"] = pd.to_numeric(visitas_F22["latitude_1"], errors="coerce")
visitas_F22["longitude_1"] = pd.to_numeric(visitas_F22["longitude_1"], errors="coerce")

# Eliminar filas incompletas
visitas_F22 = visitas_F22.dropna(
    subset=["user_id", "event_date", "latitude_1", "longitude_1"]
).copy()

visitas_F22["user_id"] = visitas_F22["user_id"].astype(int)

# 2) Crear fecha/hora en Chile y día de la semana
visitas_F22["event_date_cl"] = visitas_F22["event_date"].dt.tz_convert("America/Santiago")

dias = {
    "Monday": "lunes",
    "Tuesday": "martes",
    "Wednesday": "miércoles",
    "Thursday": "jueves",
    "Friday": "viernes",
    "Saturday": "sábado",
    "Sunday": "domingo"
}

visitas_F22["dia"] = visitas_F22["event_date_cl"].dt.day_name().map(dias)

# 3) Ordenar por usuario, día y tiempo
df = visitas_F22.sort_values(
    ["user_id", "dia", "event_date_cl"]
).reset_index(drop=True)

# 4) Función Haversine
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000  # radio de la Tierra en metros

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c

# 5) Agrupar por usuario y día usando ancla geográfica
dfs_reducidos = []

for (user_id, dia), grupo in df.groupby(["user_id", "dia"], sort=False):
    grupo = grupo.copy().reset_index(drop=False)

    parada_ids = []
    dist_anchor_list = []

    parada_actual = 0

    # Ancla inicial de la primera parada del usuario-día
    anchor_lat = grupo.loc[0, "latitude_1"]
    anchor_lon = grupo.loc[0, "longitude_1"]

    parada_ids.append(parada_actual)
    dist_anchor_list.append(0.0)

    for i in range(1, len(grupo)):
        lat_i = grupo.loc[i, "latitude_1"]
        lon_i = grupo.loc[i, "longitude_1"]

        dist_anchor = haversine_m(anchor_lat, anchor_lon, lat_i, lon_i)

        # Si sigue dentro de 10 m del ancla, misma parada
        if dist_anchor <= RANGO_PARADA_M:
            parada_ids.append(parada_actual)
            dist_anchor_list.append(dist_anchor)
        else:
            # Nueva parada
            parada_actual += 1
            anchor_lat = lat_i
            anchor_lon = lon_i

            parada_ids.append(parada_actual)
            dist_anchor_list.append(0.0)

    grupo["parada_id"] = parada_ids
    grupo["dist_anchor_m"] = dist_anchor_list

    # ID global de parada realizada considerando user_id + día
    grupo["paradas_realizadas"] = (
        grupo["user_id"].astype(str)
        + "_"
        + grupo["dia"].astype(str)
        + "_PR_"
        + grupo["parada_id"].astype(str)
    )

    # Reducir: dejar una fila representativa por parada realizada
    reducido = (
        grupo.groupby("paradas_realizadas", as_index=False)
        .agg(
            user_id=("user_id", "first"),
            dia=("dia", "first"),
            event_date_inicio=("event_date_cl", "min"),
            event_date_fin=("event_date_cl", "max"),
            latitude_1=("latitude_1", "mean"),
            longitude_1=("longitude_1", "mean"),
            n_checkouts_agrupados=("paradas_realizadas", "size")
        )
    )

    reducido["filas_eliminables"] = reducido["n_checkouts_agrupados"] - 1

    dfs_reducidos.append(reducido)

# 6) DataFrame final de paradas realizadas comprimidas
paradas_reales = pd.concat(dfs_reducidos, ignore_index=True)

# Duración de la parada realizada
paradas_reales["duracion_min"] = (
    (paradas_reales["event_date_fin"] - paradas_reales["event_date_inicio"])
    .dt.total_seconds() / 60
)

# Ordenar resultado final
paradas_reales = paradas_reales.sort_values(
    ["user_id", "dia", "event_date_inicio"]
).reset_index(drop=True)

print("Shape original visitas_F22:", visitas_F22.shape)
print("Shape reducido paradas_reales:", paradas_reales.shape)
print(paradas_reales.head())

In [ ]:
# ORDENAR GPS Y CALCULAR DISTANCIA / TIEMPO / VELOCIDAD
# Considerando ruta = owner_id + día

import pandas as pd
import numpy as np

gps_F22 = gps_22_limpio

# Normalizar tipos
gps_F22["created"] = pd.to_datetime(gps_F22["created"], errors="coerce", utc=True)
gps_F22["latitude"] = pd.to_numeric(gps_F22["latitude"], errors="coerce")
gps_F22["longitude"] = pd.to_numeric(gps_F22["longitude"], errors="coerce")

# Eliminar filas incompletas
gps_F22 = gps_F22.dropna(subset=["owner_id", "created", "latitude", "longitude"]).copy()

# Crear fecha/hora Chile y día
gps_F22["created_cl"] = gps_F22["created"].dt.tz_convert("America/Santiago")

dias = {
    "Monday": "lunes",
    "Tuesday": "martes",
    "Wednesday": "miércoles",
    "Thursday": "jueves",
    "Friday": "viernes",
    "Saturday": "sábado",
    "Sunday": "domingo"
}

gps_F22["dia"] = gps_F22["created_cl"].dt.day_name().map(dias)

# Asegurar orden correcto por ruta: owner_id + día + tiempo
gps_F22 = gps_F22.sort_values(
    ["owner_id", "dia", "created_cl"]
).reset_index(drop=True)

# Función Haversine en metros
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000  # radio de la Tierra en metros

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c

# Coordenadas anteriores dentro de cada ruta owner_id + día
gps_F22["prev_latitude"] = gps_F22.groupby(["owner_id", "dia"])["latitude"].shift(1)
gps_F22["prev_longitude"] = gps_F22.groupby(["owner_id", "dia"])["longitude"].shift(1)
gps_F22["prev_created"] = gps_F22.groupby(["owner_id", "dia"])["created_cl"].shift(1)

# Distancia entre punto actual y punto anterior dentro de la misma ruta
gps_F22["dist_prev_m"] = haversine_m(
    gps_F22["prev_latitude"],
    gps_F22["prev_longitude"],
    gps_F22["latitude"],
    gps_F22["longitude"]
)

# Diferencia de tiempo en segundos
gps_F22["time_diff_sec"] = (
    (gps_F22["created_cl"] - gps_F22["prev_created"])
    .dt.total_seconds()
)

# Diferencia de tiempo en minutos
gps_F22["time_diff_min_calc"] = gps_F22["time_diff_sec"] / 60

# Velocidad en m/s
gps_F22["speed_m_s"] = gps_F22["dist_prev_m"] / gps_F22["time_diff_sec"]

# Velocidad en km/h
gps_F22["speed_kmh"] = gps_F22["speed_m_s"] * 3.6

# Evitar infinitos cuando time_diff_sec = 0
gps_F22.loc[~np.isfinite(gps_F22["speed_m_s"]), "speed_m_s"] = np.nan
gps_F22.loc[~np.isfinite(gps_F22["speed_kmh"]), "speed_kmh"] = np.nan

print("Columnas creadas:")
print([
    "created_cl", "dia",
    "prev_latitude", "prev_longitude", "prev_created",
    "dist_prev_m", "time_diff_sec", "time_diff_min_calc",
    "speed_m_s", "speed_kmh"
])

In [ ]:
# BLOQUE 2: DETECCIÓN DE PARADA EN VENTANA DE 1 MINUTOS
# Criterio:
# si en 1 minutos el vehículo recorre menos de 100 metros, lo consideramos parada
# Unidad de análisis: ruta = owner_id + día

STOP_WINDOW_SEC = 60
STOP_MAX_DIST_M = 100

# Asegurar orden correcto por ruta
gps_F22 = gps_F22.sort_values(["owner_id", "dia", "created_cl"]).reset_index(drop=True)

# Columnas auxiliares
gps_F22["stop_ref_idx"] = np.nan
gps_F22["stop_ref_created"] = pd.NaT
gps_F22["stop_window_dist_m"] = np.nan
gps_F22["stop_window_time_sec"] = np.nan
gps_F22["is_stop_point"] = False

# Procesar por owner_id + día
for (owner_id, dia), idx in gps_F22.groupby(["owner_id", "dia"]).groups.items():
    idx = np.array(list(idx))
    grupo = gps_F22.loc[idx].copy()

    tiempos = grupo["created_cl"].values.astype("datetime64[ns]")
    latitudes = grupo["latitude"].values
    longitudes = grupo["longitude"].values

    j = 0
    stop_ref_idx = np.full(len(grupo), np.nan)
    stop_ref_created = np.full(len(grupo), np.datetime64("NaT"), dtype="datetime64[ns]")
    stop_window_dist_m = np.full(len(grupo), np.nan)
    stop_window_time_sec = np.full(len(grupo), np.nan)
    is_stop_point = np.full(len(grupo), False)

    for i in range(len(grupo)):
        if j < i + 1:
            j = i + 1

        while j < len(grupo):
            delta_sec = (tiempos[j] - tiempos[i]) / np.timedelta64(1, "s")
            if delta_sec >= STOP_WINDOW_SEC:
                break
            j += 1

        if j < len(grupo):
            delta_sec = (tiempos[j] - tiempos[i]) / np.timedelta64(1, "s")

            dist_m = haversine_m(
                latitudes[i],
                longitudes[i],
                latitudes[j],
                longitudes[j]
            )

            stop_ref_idx[i] = idx[j]
            stop_ref_created[i] = tiempos[j]
            stop_window_dist_m[i] = dist_m
            stop_window_time_sec[i] = delta_sec
            is_stop_point[i] = dist_m < STOP_MAX_DIST_M

    gps_F22.loc[idx, "stop_ref_idx"] = stop_ref_idx
    gps_F22.loc[idx, "stop_ref_created"] = stop_ref_created
    gps_F22.loc[idx, "stop_window_dist_m"] = stop_window_dist_m
    gps_F22.loc[idx, "stop_window_time_sec"] = stop_window_time_sec
    gps_F22.loc[idx, "is_stop_point"] = is_stop_point

print("Resumen de puntos evaluados como parada:")
print(gps_F22["is_stop_point"].value_counts(dropna=False))

In [ ]:
# CHEQUEO RÁPIDO

columnas_revision = [
    "owner_id",
    "created",
    "latitude",
    "longitude",
    "stop_ref_idx",
    "stop_window_time_sec",
    "stop_window_dist_m",
    "is_stop_point"
]

gps_F22[columnas_revision].head(20)

In [ ]:
# =========================================================
# BLOQUE 4: AGRUPAR PUNTOS CONSECUTIVOS DE PARADA
# Unidad de análisis: ruta = owner_id + día
# =========================================================

gps_F22 = gps_F22.copy()

# Asegurar orden correcto
gps_F22 = gps_F22.sort_values(
    ["owner_id", "dia", "created_cl"]
).reset_index(drop=True)

gps_F22["stop_state_change"] = (
    gps_F22.groupby(["owner_id", "dia"])["is_stop_point"]
    .transform(lambda s: s.ne(s.shift(1)))
)

gps_F22["stop_segment_raw"] = (
    gps_F22.groupby(["owner_id", "dia"])["stop_state_change"]
    .cumsum()
)

gps_F22["stop_segment_id"] = np.where(
    gps_F22["is_stop_point"],
    (
        gps_F22["owner_id"].astype(str)
        + "_"
        + gps_F22["dia"].astype(str)
        + "_STOP_"
        + gps_F22["stop_segment_raw"].astype(str)
    ),
    np.nan
)

print("Cantidad de segmentos detectados:")
print(gps_F22["stop_segment_id"].nunique(dropna=True))

In [ ]:
# =========================================================
# BLOQUE 5: RESUMEN DE PARADAS DETECTADAS
# =========================================================

paradas_gps_22 = (
    gps_F22[gps_F22["is_stop_point"]]
    .groupby("stop_segment_id", as_index=False)
    .agg(
        owner_id=("owner_id", "first"),
        start_time=("created", "min"),
        end_time=("created", "max"),
        n_points=("created", "size"),
        mean_latitude=("latitude", "mean"),
        mean_longitude=("longitude", "mean"),
        min_window_dist_m=("stop_window_dist_m", "min"),
        max_window_dist_m=("stop_window_dist_m", "max"),
        mean_window_dist_m=("stop_window_dist_m", "mean")
    )
)

paradas_gps_22["duration_min"] = (
    (paradas_gps_22["end_time"] - paradas_gps_22["start_time"])
    .dt.total_seconds() / 60
)

paradas_gps_22 = paradas_gps_22[
    [
        "stop_segment_id",
        "owner_id",
        "start_time",
        "end_time",
        "duration_min",
        "n_points",
        "mean_latitude",
        "mean_longitude",
        "min_window_dist_m",
        "max_window_dist_m",
        "mean_window_dist_m"
    ]
].sort_values(["owner_id", "start_time"]).reset_index(drop=True)

print("Cantidad de paradas resumidas:", len(paradas_gps_22))
paradas_gps_22.head()

In [ ]:
from keplergl import KeplerGl
import json

In [ ]:
# =========================================================
# FILTRO GENERAL PREVIO:
# 1) dejar solo rutas comunes entre gps_F22 y visita_22
# 2) calcular ventana horaria real disponible en GPS por owner_id + día
# 3) filtrar visita_22 a checkouts dentro de esa ventana
# 4) re-filtrar gps_F22 a las rutas que sobreviven
# =========================================================

gps_F22 = gps_F22.copy()
visita_22 = visitas_F22.copy()

# ---------------------------------------------------------
# 1) Normalizar IDs y timestamps
# ---------------------------------------------------------
gps_F22["owner_id"] = pd.to_numeric(gps_F22["owner_id"], errors="coerce")
visita_22["user_id"] = pd.to_numeric(visita_22["user_id"], errors="coerce")

gps_F22["created"] = pd.to_datetime(gps_F22["created"], errors="coerce", utc=True)
visita_22["event_date"] = pd.to_datetime(
    visita_22["event_date"],
    errors="coerce",
    utc=True,
    format="mixed"
)

gps_F22 = gps_F22.dropna(subset=["owner_id", "created"]).copy()
visita_22 = visita_22.dropna(subset=["user_id", "event_date"]).copy()

gps_F22["owner_id"] = gps_F22["owner_id"].astype(int)
visita_22["user_id"] = visita_22["user_id"].astype(int)

# ---------------------------------------------------------
# 2) Crear fecha/hora Chile y día
# ---------------------------------------------------------
gps_F22["created_cl"] = gps_F22["created"].dt.tz_convert("America/Santiago")
visita_22["event_date_cl"] = visita_22["event_date"].dt.tz_convert("America/Santiago")

dias = {
    "Monday": "lunes",
    "Tuesday": "martes",
    "Wednesday": "miércoles",
    "Thursday": "jueves",
    "Friday": "viernes",
    "Saturday": "sábado",
    "Sunday": "domingo"
}

gps_F22["dia"] = gps_F22["created_cl"].dt.day_name().map(dias)
visita_22["dia"] = visita_22["event_date_cl"].dt.day_name().map(dias)

# ---------------------------------------------------------
# 3) Dejar solo IDs comunes inicialmente
# ---------------------------------------------------------
ids_comunes = sorted(
    set(gps_F22["owner_id"].unique()).intersection(
        set(visita_22["user_id"].unique())
    )
)

gps_F22 = gps_F22[gps_F22["owner_id"].isin(ids_comunes)].copy()
visita_22 = visita_22[visita_22["user_id"].isin(ids_comunes)].copy()

# ---------------------------------------------------------
# 4) Calcular ventana GPS por ruta: owner_id + día
# ---------------------------------------------------------
gps_window = (
    gps_F22.groupby(["owner_id", "dia"], as_index=False)
    .agg(
        gps_start_time=("created_cl", "min"),
        gps_end_time=("created_cl", "max"),
        gps_n_points=("created_cl", "size")
    )
)

# ---------------------------------------------------------
# 5) Cruzar ventana GPS con visitas por user_id + día
# ---------------------------------------------------------
visita_22 = visita_22.merge(
    gps_window,
    left_on=["user_id", "dia"],
    right_on=["owner_id", "dia"],
    how="inner"
)

# ---------------------------------------------------------
# 6) Quedarse solo con visitas dentro de la ventana GPS del mismo día
# ---------------------------------------------------------
visita_22 = visita_22[
    (visita_22["event_date_cl"] >= visita_22["gps_start_time"]) &
    (visita_22["event_date_cl"] <= visita_22["gps_end_time"])
].copy()

# ---------------------------------------------------------
# 7) Re-filtrar gps_F22 a las rutas owner_id + día que sobrevivieron
# ---------------------------------------------------------
rutas_finales = (
    visita_22[["user_id", "dia"]]
    .drop_duplicates()
    .rename(columns={"user_id": "owner_id"})
)

gps_F22 = gps_F22.merge(
    rutas_finales.assign(ruta_valida=1),
    on=["owner_id", "dia"],
    how="inner"
).drop(columns=["ruta_valida"])

visita_22 = visita_22.merge(
    rutas_finales.rename(columns={"owner_id": "user_id"}).assign(ruta_valida=1),
    on=["user_id", "dia"],
    how="inner"
).drop(columns=["ruta_valida"])

# ---------------------------------------------------------
# 8) Limpieza final y orden
# ---------------------------------------------------------
if "owner_id_y" in visita_22.columns:
    visita_22 = visita_22.drop(columns=["owner_id_y"])

if "owner_id_x" in visita_22.columns:
    visita_22 = visita_22.rename(columns={"owner_id_x": "owner_id"})

gps_F22 = gps_F22.sort_values(["owner_id", "dia", "created_cl"]).reset_index(drop=True)
visita_22 = visita_22.sort_values(["user_id", "dia", "event_date_cl"]).reset_index(drop=True)

print("=======================================")
print(f"IDs comunes iniciales: {len(ids_comunes)}")
print(f"Rutas finales owner_id + día con visitas dentro de ventana GPS: {len(rutas_finales)}")
print(f"gps_F22 shape final: {gps_F22.shape}")
print(f"visita_22 shape final: {visita_22.shape}")
print(f"owner_id únicos en gps_F22: {gps_F22['owner_id'].nunique()}")
print(f"user_id únicos en visita_22: {visita_22['user_id'].nunique()}")

print("\nMuestra de ventanas GPS por owner_id + día:")
print(gps_window.head())

In [ ]:
# =========================================================
# BLOQUE 1: PREPARAR CAPAS PARA KEPLERGL
# Considerando ruta = owner_id/user_id + día
# =========================================================

# 1) GPS completo
gps_map = gps_F22.copy()

gps_map["created"] = pd.to_datetime(gps_map["created"], errors="coerce", utc=True)

if "created_cl" not in gps_map.columns:
    gps_map["created_cl"] = gps_map["created"].dt.tz_convert("America/Santiago")

if "dia" not in gps_map.columns:
    dias = {
        "Monday": "lunes",
        "Tuesday": "martes",
        "Wednesday": "miércoles",
        "Thursday": "jueves",
        "Friday": "viernes",
        "Saturday": "sábado",
        "Sunday": "domingo"
    }
    gps_map["dia"] = gps_map["created_cl"].dt.day_name().map(dias)

gps_map = gps_map.dropna(subset=["owner_id", "dia", "latitude", "longitude"]).copy()
gps_map["tipo_capa"] = "gps_completo"

gps_map = gps_map.sort_values(
    ["owner_id", "dia", "created_cl"]
).reset_index(drop=True)


# 2) Puntos detectados como parada según el criterio actual
gps_stops_map = gps_F22[gps_F22["is_stop_point"] == True].copy()

gps_stops_map["created"] = pd.to_datetime(gps_stops_map["created"], errors="coerce", utc=True)

if "created_cl" not in gps_stops_map.columns:
    gps_stops_map["created_cl"] = gps_stops_map["created"].dt.tz_convert("America/Santiago")

if "dia" not in gps_stops_map.columns:
    gps_stops_map["dia"] = gps_stops_map["created_cl"].dt.day_name().map(dias)

gps_stops_map = gps_stops_map.dropna(subset=["owner_id", "dia", "latitude", "longitude"]).copy()
gps_stops_map["tipo_capa"] = "parada_detectada_gps"

gps_stops_map = gps_stops_map.sort_values(
    ["owner_id", "dia", "created_cl"]
).reset_index(drop=True)


# 3) Paradas realizadas desde visita_22
visitas_map = visita_22.copy()

visitas_map["user_id"] = pd.to_numeric(visitas_map["user_id"], errors="coerce")
visitas_map["event_date"] = pd.to_datetime(
    visitas_map["event_date"],
    errors="coerce",
    utc=True,
    format="mixed"
)

visitas_map = visitas_map.dropna(
    subset=["user_id", "event_date", "latitude_1", "longitude_1"]
).copy()

visitas_map["user_id"] = visitas_map["user_id"].astype(int)

if "event_date_cl" not in visitas_map.columns:
    visitas_map["event_date_cl"] = visitas_map["event_date"].dt.tz_convert("America/Santiago")

if "dia" not in visitas_map.columns:
    visitas_map["dia"] = visitas_map["event_date_cl"].dt.day_name().map(dias)

visitas_map["latitude_1"] = pd.to_numeric(visitas_map["latitude_1"], errors="coerce")
visitas_map["longitude_1"] = pd.to_numeric(visitas_map["longitude_1"], errors="coerce")

visitas_map = visitas_map.dropna(
    subset=["user_id", "dia", "latitude_1", "longitude_1", "event_date_cl"]
).copy()

visitas_map["tipo_capa"] = "parada_realizada"

visitas_map = visitas_map.sort_values(
    ["user_id", "dia", "event_date_cl"]
).reset_index(drop=True)


print(visitas_map[["user_id", "dia", "event_date_cl", "latitude_1", "longitude_1"]].head())

print("gps_map:", gps_map.shape)
print("gps_stops_map:", gps_stops_map.shape)
print("visitas_map:", visitas_map.shape)

In [ ]:
# =========================================================
# BLOQUE 2: CREAR MAPA KEPLERGL CON 3 CAPAS
# =========================================================

#mapa = KeplerGl(height=700)

#mapa.add_data(data=gps_map, name="gps_completo")
#mapa.add_data(data=gps_stops_map, name="paradas_detectadas_gps")
#mapa.add_data(data=visitas_map, name="paradas_realizadas")

#mapa

In [ ]:
# =========================================================
# AGRUPAR REALIZADAS, AGRUPAR DETECTADAS, HACER MATCH Y
# DEVOLVER TABLA RESUMEN
# Considerando usuario + dia para evitar cruces entre días
# =========================================================

import pandas as pd
import numpy as np

RADIO_AGRUPACION_M = 50
RADIO_MATCH_M = 50
DIA_COL = "dia"

# ---------------------------------------------------------
# 1) Función Haversine
# ---------------------------------------------------------
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000  # metros

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c


# ---------------------------------------------------------
# 2) Función genérica para agrupar paradas por usuario + día
# ---------------------------------------------------------
def agrupar_paradas_por_ancla(
    df,
    user_col,
    time_col,
    lat_col,
    lon_col,
    radio_m,
    prefix_id,
    dia_col="dia"
):
    work = df.copy()

    work[user_col] = pd.to_numeric(work[user_col], errors="coerce")
    work[time_col] = pd.to_datetime(work[time_col], errors="coerce", utc=True)
    work[lat_col] = pd.to_numeric(work[lat_col], errors="coerce")
    work[lon_col] = pd.to_numeric(work[lon_col], errors="coerce")

    if dia_col not in work.columns:
        work[dia_col] = work[time_col].dt.day_name()

    work[dia_col] = work[dia_col].astype(str).str.lower().str.strip()

    work = work.dropna(subset=[user_col, time_col, lat_col, lon_col, dia_col]).copy()
    work[user_col] = work[user_col].astype(int)

    work = work.sort_values([user_col, dia_col, time_col]).reset_index(drop=True)

    grupos_out = []

    for (uid, dia), grupo in work.groupby([user_col, dia_col], sort=False):
        grupo = grupo.copy().reset_index(drop=True)

        anchor_lat = grupo.loc[0, lat_col]
        anchor_lon = grupo.loc[0, lon_col]
        cluster_id_actual = 0

        cluster_ids = [cluster_id_actual]
        dist_anchor_list = [0.0]

        for i in range(1, len(grupo)):
            lat_i = grupo.loc[i, lat_col]
            lon_i = grupo.loc[i, lon_col]

            dist_anchor = haversine_m(anchor_lat, anchor_lon, lat_i, lon_i)

            if dist_anchor <= radio_m:
                cluster_ids.append(cluster_id_actual)
                dist_anchor_list.append(float(dist_anchor))
            else:
                cluster_id_actual += 1
                anchor_lat = lat_i
                anchor_lon = lon_i

                cluster_ids.append(cluster_id_actual)
                dist_anchor_list.append(0.0)

        grupo["cluster_local"] = cluster_ids
        grupo["dist_anchor_m"] = dist_anchor_list

        grupo["cluster_id"] = (
            grupo[user_col].astype(str)
            + "_" + grupo[dia_col].astype(str)
            + f"_{prefix_id}_"
            + grupo["cluster_local"].astype(str)
        )

        resumen = (
            grupo.groupby("cluster_id", as_index=False)
            .agg(
                usuario=(user_col, "first"),
                dia=(dia_col, "first"),
                inicio=(time_col, "min"),
                fin=(time_col, "max"),
                lat_agrupada=(lat_col, "mean"),
                lon_agrupada=(lon_col, "mean"),
                n_filas_agrupadas=("cluster_id", "size")
            )
        )

        grupos_out.append(resumen)

    if len(grupos_out) == 0:
        return pd.DataFrame(columns=[
            "cluster_id", "usuario", "dia", "inicio", "fin",
            "lat_agrupada", "lon_agrupada", "n_filas_agrupadas"
        ])

    agrupado = pd.concat(grupos_out, ignore_index=True)

    agrupado["duracion_min"] = (
        (agrupado["fin"] - agrupado["inicio"]).dt.total_seconds() / 60
    )

    agrupado = agrupado.sort_values(["usuario", "dia", "inicio"]).reset_index(drop=True)

    return agrupado


# ---------------------------------------------------------
# 3) AGRUPADAS REALIZADAS
# ---------------------------------------------------------
paradas_agrupadas_realizadas = agrupar_paradas_por_ancla(
    df=visitas_map,
    user_col="user_id",
    time_col="event_date",
    lat_col="latitude_1",
    lon_col="longitude_1",
    radio_m=RADIO_AGRUPACION_M,
    prefix_id="REAL",
    dia_col=DIA_COL
)

paradas_agrupadas_realizadas = paradas_agrupadas_realizadas.rename(columns={
    "cluster_id": "parada_agrupada_realizada_id",
    "usuario": "user_id",
    "inicio": "event_date_inicio",
    "fin": "event_date_fin",
    "lat_agrupada": "latitude_1",
    "lon_agrupada": "longitude_1"
})

print("paradas_agrupadas_realizadas:", paradas_agrupadas_realizadas.shape)
display(paradas_agrupadas_realizadas.head())


# ---------------------------------------------------------
# 4) AGRUPADAS DETECTADAS
# ---------------------------------------------------------
paradas_agrupadas_detectadas = agrupar_paradas_por_ancla(
    df=gps_stops_map,
    user_col="owner_id",
    time_col="created",
    lat_col="latitude",
    lon_col="longitude",
    radio_m=RADIO_AGRUPACION_M,
    prefix_id="DET",
    dia_col=DIA_COL
)

paradas_agrupadas_detectadas = paradas_agrupadas_detectadas.rename(columns={
    "cluster_id": "parada_agrupada_detectada_id",
    "usuario": "owner_id",
    "inicio": "created_inicio",
    "fin": "created_fin",
    "lat_agrupada": "latitude",
    "lon_agrupada": "longitude"
})

print("paradas_agrupadas_detectadas:", paradas_agrupadas_detectadas.shape)
display(paradas_agrupadas_detectadas.head())


# ---------------------------------------------------------
# 5) MATCH 1 A 1 entre AGRUPADAS REALIZADAS y DETECTADAS
#    mismo usuario + mismo día + radio <= 50m
# ---------------------------------------------------------
def match_agrupadas_realizadas_detectadas(
    realizadas_df,
    detectadas_df,
    match_radius_m=50,
    dia_col="dia"
):
    real = realizadas_df.copy().reset_index(drop=True)
    det = detectadas_df.copy().reset_index(drop=True)

    if real.empty or det.empty:
        return pd.DataFrame(columns=[
            "parada_agrupada_realizada_id",
            "parada_agrupada_detectada_id",
            "dia",
            "user_id",
            "owner_id",
            "real_latitude_1",
            "real_longitude_1",
            "det_latitude",
            "det_longitude",
            "distance_m",
            "event_date_inicio",
            "event_date_fin",
            "created_inicio",
            "created_fin"
        ])

    real[dia_col] = real[dia_col].astype(str).str.lower().str.strip()
    det[dia_col] = det[dia_col].astype(str).str.lower().str.strip()

    candidates = []

    det_by_user_day = {
        (int(uid), dia): g.copy()
        for (uid, dia), g in det.groupby(["owner_id", dia_col], sort=False)
    }

    for _, r in real.iterrows():
        uid = int(r["user_id"])
        dia = r[dia_col]

        key = (uid, dia)

        if key not in det_by_user_day:
            continue

        cand = det_by_user_day[key]

        dists = haversine_m(
            r["latitude_1"],
            r["longitude_1"],
            cand["latitude"].values,
            cand["longitude"].values
        )

        ok = dists <= match_radius_m

        if np.any(ok):
            cand_ok = cand.loc[ok].copy()
            cand_ok["distance_m"] = dists[ok]

            for _, d in cand_ok.iterrows():
                candidates.append({
                    "parada_agrupada_realizada_id": r["parada_agrupada_realizada_id"],
                    "parada_agrupada_detectada_id": d["parada_agrupada_detectada_id"],
                    "dia": dia,
                    "user_id": r["user_id"],
                    "owner_id": d["owner_id"],
                    "real_latitude_1": r["latitude_1"],
                    "real_longitude_1": r["longitude_1"],
                    "det_latitude": d["latitude"],
                    "det_longitude": d["longitude"],
                    "distance_m": float(d["distance_m"]),
                    "event_date_inicio": r["event_date_inicio"],
                    "event_date_fin": r["event_date_fin"],
                    "created_inicio": d["created_inicio"],
                    "created_fin": d["created_fin"],
                })

    candidates_df = pd.DataFrame(candidates)

    if candidates_df.empty:
        return candidates_df

    candidates_df = candidates_df.sort_values("distance_m").reset_index(drop=True)

    used_real = set()
    used_det = set()
    selected = []

    for _, row in candidates_df.iterrows():
        real_id = row["parada_agrupada_realizada_id"]
        det_id = row["parada_agrupada_detectada_id"]

        if real_id in used_real:
            continue

        if det_id in used_det:
            continue

        used_real.add(real_id)
        used_det.add(det_id)
        selected.append(row)

    matched = pd.DataFrame(selected).reset_index(drop=True)

    return matched


paradas_agrupadas_match = match_agrupadas_realizadas_detectadas(
    realizadas_df=paradas_agrupadas_realizadas,
    detectadas_df=paradas_agrupadas_detectadas,
    match_radius_m=RADIO_MATCH_M,
    dia_col=DIA_COL
)

print("paradas_agrupadas_match:", paradas_agrupadas_match.shape)
display(paradas_agrupadas_match.head())


# ---------------------------------------------------------
# 6) TABLA RESUMEN GLOBAL
# ---------------------------------------------------------
tabla_resumen_global = pd.DataFrame([{
    "paradas_agrupadas_realizadas": paradas_agrupadas_realizadas["parada_agrupada_realizada_id"].nunique(),
    "paradas_agrupadas_detectadas": paradas_agrupadas_detectadas["parada_agrupada_detectada_id"].nunique(),
    "paradas_agrupadas_match": paradas_agrupadas_match["parada_agrupada_realizada_id"].nunique()
}])

display(tabla_resumen_global)


# ---------------------------------------------------------
# 7) TABLA RESUMEN POR USUARIO Y DÍA
# ---------------------------------------------------------
real_user_day = (
    paradas_agrupadas_realizadas.groupby(["user_id", "dia"], as_index=False)
    .agg(paradas_agrupadas_realizadas=("parada_agrupada_realizada_id", "nunique"))
    .rename(columns={"user_id": "usuario"})
)

det_user_day = (
    paradas_agrupadas_detectadas.groupby(["owner_id", "dia"], as_index=False)
    .agg(paradas_agrupadas_detectadas=("parada_agrupada_detectada_id", "nunique"))
    .rename(columns={"owner_id": "usuario"})
)

if not paradas_agrupadas_match.empty:
    match_user_day = (
        paradas_agrupadas_match.groupby(["user_id", "dia"], as_index=False)
        .agg(paradas_agrupadas_match=("parada_agrupada_realizada_id", "nunique"))
        .rename(columns={"user_id": "usuario"})
    )
else:
    match_user_day = pd.DataFrame(columns=["usuario", "dia", "paradas_agrupadas_match"])

tabla_resumen_usuario_dia = real_user_day.merge(
    det_user_day,
    on=["usuario", "dia"],
    how="outer"
)

tabla_resumen_usuario_dia = tabla_resumen_usuario_dia.merge(
    match_user_day,
    on=["usuario", "dia"],
    how="outer"
)

tabla_resumen_usuario_dia["paradas_agrupadas_realizadas"] = (
    tabla_resumen_usuario_dia["paradas_agrupadas_realizadas"].fillna(0).astype(int)
)

tabla_resumen_usuario_dia["paradas_agrupadas_detectadas"] = (
    tabla_resumen_usuario_dia["paradas_agrupadas_detectadas"].fillna(0).astype(int)
)

tabla_resumen_usuario_dia["paradas_agrupadas_match"] = (
    tabla_resumen_usuario_dia["paradas_agrupadas_match"].fillna(0).astype(int)
)

tabla_resumen_usuario_dia = tabla_resumen_usuario_dia.sort_values(
    ["paradas_agrupadas_match", "paradas_agrupadas_realizadas", "paradas_agrupadas_detectadas"],
    ascending=[False, False, False]
).reset_index(drop=True)

display(tabla_resumen_usuario_dia.head(50))


# ---------------------------------------------------------
# 8) TABLA RESUMEN POR USUARIO, SIN SEPARAR DÍA
# ---------------------------------------------------------
tabla_resumen_usuario = (
    tabla_resumen_usuario_dia
    .groupby("usuario", as_index=False)
    .agg(
        paradas_agrupadas_realizadas=("paradas_agrupadas_realizadas", "sum"),
        paradas_agrupadas_detectadas=("paradas_agrupadas_detectadas", "sum"),
        paradas_agrupadas_match=("paradas_agrupadas_match", "sum")
    )
    .sort_values(
        ["paradas_agrupadas_match", "paradas_agrupadas_realizadas", "paradas_agrupadas_detectadas"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

display(tabla_resumen_usuario.head(50))

In [ ]:
# =========================================================
# 1) PARADAS DETECTADAS QUE SÍ HICIERON MATCH
#    Congruente con separación por día
# =========================================================

paradas_detectadas_match = paradas_agrupadas_detectadas.merge(
    paradas_agrupadas_match[[
        "parada_agrupada_detectada_id",
        "parada_agrupada_realizada_id",
        "user_id",
        "dia"
    ]],
    on=["parada_agrupada_detectada_id", "dia"],
    how="inner"
).copy()

# asegurar tipos fecha
paradas_detectadas_match["created_inicio"] = pd.to_datetime(
    paradas_detectadas_match["created_inicio"],
    errors="coerce"
)

paradas_detectadas_match["created_fin"] = pd.to_datetime(
    paradas_detectadas_match["created_fin"],
    errors="coerce"
)

# recalcular duración por seguridad
paradas_detectadas_match["duracion_min"] = (
    (paradas_detectadas_match["created_fin"] - paradas_detectadas_match["created_inicio"])
    .dt.total_seconds() / 60
)

paradas_detectadas_match = paradas_detectadas_match.dropna(
    subset=["duracion_min"]
).copy()

paradas_detectadas_match = paradas_detectadas_match[
    paradas_detectadas_match["duracion_min"] >= 0
].copy()

print("Cantidad de paradas detectadas con match:", len(paradas_detectadas_match))
display(paradas_detectadas_match.head())

In [ ]:
resumen_tiempo_match_dia = (
    paradas_detectadas_match
    .groupby("dia", as_index=False)
    .agg(
        n_paradas_match=("duracion_min", "size"),
        tiempo_promedio_min=("duracion_min", "mean"),
        tiempo_minimo_min=("duracion_min", "min"),
        tiempo_maximo_min=("duracion_min", "max"),
        tiempo_mediana_min=("duracion_min", "median"),
        std_min=("duracion_min", "std")
    )
)

display(resumen_tiempo_match_dia)

In [ ]:
# =========================================================
# 3) RESUMEN POR USUARIO + DÍA + DISTRIBUCIÓN
# =========================================================

resumen_tiempo_match_usuario_dia = (
    paradas_detectadas_match
    .groupby(["owner_id", "dia"], as_index=False)
    .agg(
        n_paradas_match=("parada_agrupada_detectada_id", "nunique"),
        tiempo_promedio_min=("duracion_min", "mean"),
        tiempo_minimo_min=("duracion_min", "min"),
        tiempo_maximo_min=("duracion_min", "max"),
        tiempo_mediana_min=("duracion_min", "median"),
        std_min=("duracion_min", "std")
    )
    .sort_values(["owner_id", "dia", "tiempo_promedio_min"], ascending=[True, True, False])
    .reset_index(drop=True)
)

display(resumen_tiempo_match_usuario_dia.head(20))

In [ ]:
# =========================================================
# 4) PARADAS DETECTADAS CON MATCH QUE QUEDARON EN 0 MINUTOS
#    Separado por día
# =========================================================

paradas_cero = paradas_detectadas_match[
    paradas_detectadas_match["duracion_min"] == 0
].copy()

print("Cantidad de paradas con duración 0:", len(paradas_cero))

display(
    paradas_cero[[
        "parada_agrupada_detectada_id",
        "owner_id",
        "dia",
        "created_inicio",
        "created_fin",
        "n_filas_agrupadas",
        "duracion_min"
    ]].head(20)
)

print("\nDistribución de n_filas_agrupadas en paradas con duración 0:")
print(paradas_cero["n_filas_agrupadas"].value_counts(dropna=False).sort_index())

print("\nDistribución de paradas con duración 0 por día:")
print(paradas_cero["dia"].value_counts(dropna=False).sort_index())

print("\nDistribución de n_filas_agrupadas por día en paradas con duración 0:")
display(
    paradas_cero
    .groupby(["dia", "n_filas_agrupadas"], as_index=False)
    .agg(n_paradas_cero=("parada_agrupada_detectada_id", "nunique"))
    .sort_values(["dia", "n_filas_agrupadas"])
    .reset_index(drop=True)
)

In [ ]:
# =========================================================
# 1) PARADAS DETECTADAS CON MATCH + DURACIÓN OBSERVADA Y ESTIMADA
#    Congruente con separación por día
# =========================================================

import pandas as pd
import numpy as np

paradas_detectadas_match = (
    paradas_agrupadas_detectadas
    .merge(
        paradas_agrupadas_match[[
            "parada_agrupada_detectada_id",
            "parada_agrupada_realizada_id",
            "user_id",
            "dia"
        ]],
        on=["parada_agrupada_detectada_id", "dia"],
        how="inner"
    )
    .copy()
)

# asegurar tipos datetime
paradas_detectadas_match["created_inicio"] = pd.to_datetime(
    paradas_detectadas_match["created_inicio"],
    errors="coerce"
)

paradas_detectadas_match["created_fin"] = pd.to_datetime(
    paradas_detectadas_match["created_fin"],
    errors="coerce"
)

# duración observada: max(created) - min(created)
paradas_detectadas_match["duracion_observada_min"] = (
    (
        paradas_detectadas_match["created_fin"] -
        paradas_detectadas_match["created_inicio"]
    )
    .dt.total_seconds() / 60
)

# limpiar casos inválidos
paradas_detectadas_match = paradas_detectadas_match.dropna(
    subset=["duracion_observada_min"]
).copy()

paradas_detectadas_match = paradas_detectadas_match[
    paradas_detectadas_match["duracion_observada_min"] >= 0
].copy()

# duración estimada:
# se suma la ventana mínima usada para declarar una parada
paradas_detectadas_match["duracion_estimada_min"] = (
    paradas_detectadas_match["duracion_observada_min"] +
    STOP_WINDOW_SEC / 60
)

# Para mantener compatibilidad con bloques posteriores
# puedes definir duracion_min como la versión que vas a analizar.
# Recomiendo usar la estimada si quieres evitar que las paradas de una fila queden en 0.
paradas_detectadas_match["duracion_min"] = paradas_detectadas_match["duracion_estimada_min"]

print("Cantidad de paradas detectadas con match:", len(paradas_detectadas_match))

display(
    paradas_detectadas_match[[
        "parada_agrupada_detectada_id",
        "parada_agrupada_realizada_id",
        "owner_id",
        "user_id",
        "dia",
        "n_filas_agrupadas",
        "created_inicio",
        "created_fin",
        "duracion_observada_min",
        "duracion_estimada_min",
        "duracion_min"
    ]].head(20)
)

## Ahora agregamos la variable comuna al gps y visitas

In [ ]:
df = paradas_detectadas_match.copy()

print("Antes:", len(df))
print("Ceros:", (df["duracion_estimada_min"] == 0).sum())

df_filtrado = df[df["duracion_estimada_min"] != 0].copy()

print("Después:", len(df_filtrado))

In [ ]:
df_filtrado = df_filtrado.copy()

df_filtrado["bloque_horario"] = df_filtrado["created_inicio"].dt.hour

In [ ]:
!pip install marginaleffects

In [ ]:
!pip install geopandas shapely pyogrio

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

def agregar_comuna_desde_coordenadas(
    df: pd.DataFrame,
    path_comunas: str,
    lat_col: str = "latitude",
    lon_col: str = "longitude",
    comuna_col_salida: str = "COMUNA"
) -> pd.DataFrame:
    """
    Agrega la columna COMUNA a un DataFrame usando lat/lon y polígonos comunales de Chile.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame original con columnas de latitud y longitud.
    path_comunas : str
        Ruta al shapefile / geojson de comunas de Chile.
    lat_col : str
        Nombre de la columna de latitud.
    lon_col : str
        Nombre de la columna de longitud.
    comuna_col_salida : str
        Nombre de la columna de salida para la comuna.

    Retorna
    -------
    pd.DataFrame
        DataFrame original + columna COMUNA.
    """

    df_out = df.copy()

    # 1) Limpiar coordenadas
    df_out[lat_col] = pd.to_numeric(df_out[lat_col], errors="coerce")
    df_out[lon_col] = pd.to_numeric(df_out[lon_col], errors="coerce")

    # Filtro geográfico básico para Chile continental + extremos razonables
    mask_valid = (
        df_out[lat_col].between(-57, -17) &
        df_out[lon_col].between(-76, -66)
    )

    # 2) Crear GeoDataFrame de puntos
    gdf_puntos = gpd.GeoDataFrame(
        df_out.loc[mask_valid].copy(),
        geometry=gpd.points_from_xy(
            df_out.loc[mask_valid, lon_col],
            df_out.loc[mask_valid, lat_col]
        ),
        crs="EPSG:4326"
    )

    # 3) Leer polígonos de comunas
    comunas = gpd.read_file(path_comunas)

    # 4) Detectar columna de nombre de comuna
    #    Esto depende del archivo que descargues
    posibles_cols = [
        "COMUNA", "Comuna", "NOM_COM", "NOM_COMUNA", "NOMBRE_COM", "nom_com"
    ]

    col_comuna = None
    for c in posibles_cols:
        if c in comunas.columns:
            col_comuna = c
            break

    if col_comuna is None:
        raise ValueError(
            f"No encontré una columna de nombre de comuna en el archivo. "
            f"Columnas disponibles: {list(comunas.columns)}"
        )

    # 5) Reproyectar si hace falta
    if comunas.crs is None:
        raise ValueError("El archivo de comunas no trae CRS definido.")
    comunas = comunas.to_crs("EPSG:4326")

    # 6) Spatial join: cada punto cae dentro de una comuna
    joined = gpd.sjoin(
        gdf_puntos,
        comunas[[col_comuna, "geometry"]],
        how="left",
        predicate="within"
    )

    # 7) Llevar resultado de vuelta al DataFrame original
    df_out[comuna_col_salida] = pd.NA
    df_out.loc[joined.index, comuna_col_salida] = joined[col_comuna].values

    return df_out

In [ ]:
df_con_comuna = agregar_comuna_desde_coordenadas(
    df=df_filtrado,
    path_comunas="comunas.shp",
    lat_col="latitude",
    lon_col="longitude",
    comuna_col_salida="Comuna"
)

In [ ]:
df_con_comuna

In [ ]:
df_con_comuna = df_con_comuna.rename(
    columns={"n_filas_agrupadas": "cantidad_paquetes"}
)

In [ ]:
df_con_comuna

In [ ]:
## Agregamos columna número de paradas por ruta por día

df_con_comuna = df_con_comuna.copy()

df_con_comuna["cantidad_paradas_owner_dia"] = (
    df_con_comuna
    .groupby(["owner_id", "dia"])["parada_agrupada_detectada_id"]
    .transform("nunique")
)


## Agregamos la limpieza de datos conversada la reunión 29/04

In [ ]:
df_con_comuna

In [ ]:
## Generamos un split - datos entre 3-8am y datos entre 9-2am

datos_madrugada = df_con_comuna[
    df_con_comuna["bloque_horario"].between(3, 8)
].copy()

datos_diarios = df_con_comuna[
    (df_con_comuna["bloque_horario"] >= 9) |
    (df_con_comuna["bloque_horario"] <= 2)
].copy()

In [ ]:
datos_madrugada["std_duracion_por_hora"] = (
    datos_madrugada
    .groupby("bloque_horario")["duracion_estimada_min"]
    .transform("std")
)

datos_diarios["std_duracion_por_hora"] = (
    datos_diarios
    .groupby("bloque_horario")["duracion_estimada_min"]
    .transform("std")
)

In [ ]:
df_con_comuna["mean_duracion_por_hora"] = (
    df_con_comuna
    .groupby("bloque_horario")["duracion_estimada_min"]
    .transform("mean")
)

In [ ]:
import pandas as pd

datos_madrugada = datos_madrugada.copy()

datos_madrugada["duracion_estimada_min"] = pd.to_numeric(
    datos_madrugada["duracion_estimada_min"],
    errors="coerce"
)

n_original = len(datos_madrugada)

datos_madrugada["mean_hora"] = (
    datos_madrugada
    .groupby("bloque_horario")["duracion_estimada_min"]
    .transform("mean")
)

datos_madrugada["std_hora"] = (
    datos_madrugada
    .groupby("bloque_horario")["duracion_estimada_min"]
    .transform("std")
    .fillna(0)
)

datos_madrugada_sin_outliers_std = datos_madrugada[
    (
        datos_madrugada["duracion_estimada_min"] >=
        datos_madrugada["mean_hora"] - 3 * datos_madrugada["std_hora"]
    )
    &
    (
        datos_madrugada["duracion_estimada_min"] <=
        datos_madrugada["mean_hora"] + 3 * datos_madrugada["std_hora"]
    )
].copy()

n_actual = len(datos_madrugada_sin_outliers_std)
n_eliminados = n_original - n_actual
pct_disminucion = (n_eliminados / n_original) * 100 if n_original > 0 else 0

print("=== MADRUGADA | Criterio 3 desviaciones estándar por hora ===")
print(f"Datos originales: {n_original:,}")
print(f"Datos eliminados como outliers: {n_eliminados:,}")
print(f"Datos actuales: {n_actual:,}")
print(f"% disminución: {pct_disminucion:.2f}%")

In [ ]:
import pandas as pd

datos_diarios = datos_diarios.copy()

datos_diarios["duracion_estimada_min"] = pd.to_numeric(
    datos_diarios["duracion_estimada_min"],
    errors="coerce"
)

n_original = len(datos_diarios)

datos_diarios["mean_hora"] = (
    datos_diarios
    .groupby("bloque_horario")["duracion_estimada_min"]
    .transform("mean")
)

datos_diarios["std_hora"] = (
    datos_diarios
    .groupby("bloque_horario")["duracion_estimada_min"]
    .transform("std")
    .fillna(0)
)

datos_diarios_sin_outliers_std = datos_diarios[
    (
        datos_diarios["duracion_estimada_min"] >=
        datos_diarios["mean_hora"] - 3 * datos_diarios["std_hora"]
    )
    &
    (
        datos_diarios["duracion_estimada_min"] <=
        datos_diarios["mean_hora"] + 3 * datos_diarios["std_hora"]
    )
].copy()

n_actual = len(datos_diarios_sin_outliers_std)
n_eliminados = n_original - n_actual
pct_disminucion = (n_eliminados / n_original) * 100 if n_original > 0 else 0

print("=== DIARIOS | Criterio 3 desviaciones estándar por hora ===")
print(f"Datos originales: {n_original:,}")
print(f"Datos eliminados como outliers: {n_eliminados:,}")
print(f"Datos actuales: {n_actual:,}")
print(f"% disminución: {pct_disminucion:.2f}%")

### Visualización de datos outliers


In [ ]:
outliers_diarios = datos_diarios.loc[
    ~datos_diarios.index.isin(datos_diarios_sin_outliers_std.index)
].copy()

print(f"Cantidad de outliers diarios: {len(outliers_diarios)}")

owners_outliers_diarios = outliers_diarios["owner_id"].unique()
print(f"Cantidad de conductores con outliers (diarios): {len(owners_outliers_diarios)}")

In [ ]:
from keplergl import KeplerGl

cols = [
    "owner_id",
    "latitude",
    "longitude",
    "duracion_estimada_min",
    "bloque_horario",
    "Comuna",
    "dia", 
    "cantidad_paquetes",
    "cantidad_paradas_owner_dia"
]

outliers_plot_diarios = outliers_diarios[cols].copy()
normales_plot_diarios = datos_diarios_sin_outliers_std[cols].copy()

In [ ]:
mapa_diarios = KeplerGl(height=600)

mapa_diarios.add_data(
    data=normales_plot_diarios,
    name="Normales Diarios"
)

mapa_diarios.add_data(
    data=outliers_plot_diarios,
    name="Outliers Diarios"
)

mapa_diarios

## Generamos algunos gráficos descriptivos 

In [ ]:
## Gráfico tiempo_serv_prom por cantidad de paradas en la ruta 
import matplotlib.pyplot as plt
import pandas as pd

df_plot = datos_diarios.copy()  # o df_con_comuna / datos_madrugada

df_plot["cantidad_paradas_owner_dia"] = pd.to_numeric(
    df_plot["cantidad_paradas_owner_dia"], errors="coerce"
)

df_plot["duracion_estimada_min"] = pd.to_numeric(
    df_plot["duracion_estimada_min"], errors="coerce"
)

df_plot = df_plot.dropna(
    subset=["cantidad_paradas_owner_dia", "duracion_estimada_min"]
)

plt.figure(figsize=(10, 6))

plt.scatter(
    df_plot["cantidad_paradas_owner_dia"],
    df_plot["duracion_estimada_min"],
    alpha=0.4
)

plt.title("Cantidad de paradas por conductor/día vs duración estimada")
plt.xlabel("Cantidad de paradas del conductor en el día")
plt.ylabel("Duración estimada de la parada (min)")
plt.grid(True, alpha=0.3)

plt.show()

### Probamos criterio de cuartiles para eliminar outliers (no utilizamos este criterio)

In [ ]:
datos_diarios = datos_diarios.copy()

# Asegurar tipo numérico
datos_diarios["duracion_estimada_min"] = pd.to_numeric(
    datos_diarios["duracion_estimada_min"],
    errors="coerce"
)

n_original = len(datos_diarios)

# Calcular Q1 y Q3 por hora
datos_diarios["Q1_hora"] = (
    datos_diarios
    .groupby("bloque_horario")["duracion_estimada_min"]
    .transform(lambda x: x.quantile(0.25))
)

datos_diarios["Q3_hora"] = (
    datos_diarios
    .groupby("bloque_horario")["duracion_estimada_min"]
    .transform(lambda x: x.quantile(0.75))
)

# IQR
datos_diarios["IQR_hora"] = (
    datos_diarios["Q3_hora"] - datos_diarios["Q1_hora"]
).fillna(0)

# Límites
datos_diarios["lim_inf"] = datos_diarios["Q1_hora"] - 1.5 * datos_diarios["IQR_hora"]
datos_diarios["lim_sup"] = datos_diarios["Q3_hora"] + 1.5 * datos_diarios["IQR_hora"]

# Filtrar
datos_diarios_sin_outliers_iqr = datos_diarios[
    (datos_diarios["duracion_estimada_min"] >= datos_diarios["lim_inf"]) &
    (datos_diarios["duracion_estimada_min"] <= datos_diarios["lim_sup"])
].copy()

# Métricas
n_actual = len(datos_diarios_sin_outliers_iqr)
n_eliminados = n_original - n_actual
pct = (n_eliminados / n_original) * 100 if n_original > 0 else 0

print("=== DIARIOS | IQR por hora ===")
print(f"Datos originales: {n_original:,}")
print(f"Datos eliminados: {n_eliminados:,}")
print(f"Datos actuales: {n_actual:,}")
print(f"% disminución: {pct:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df_plot = datos_diarios_sin_outliers_iqr.copy()

df_plot["cantidad_paradas_owner_dia"] = pd.to_numeric(
    df_plot["cantidad_paradas_owner_dia"], errors="coerce"
)

df_plot["duracion_estimada_min"] = pd.to_numeric(
    df_plot["duracion_estimada_min"], errors="coerce"
)

df_plot = df_plot.dropna(
    subset=["cantidad_paradas_owner_dia", "duracion_estimada_min"]
)

plt.figure(figsize=(10, 6))

plt.scatter(
    df_plot["cantidad_paradas_owner_dia"],
    df_plot["duracion_estimada_min"],
    alpha=0.4
)

plt.title("Relación entre cantidad de paradas y duración (sin outliers - IQR)")
plt.xlabel("Cantidad de paradas por conductor/día")
plt.ylabel("Duración estimada (min)")
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
df_promedio = (
    df_plot
    .groupby("cantidad_paradas_owner_dia", as_index=False)["duracion_estimada_min"]
    .mean()
)

plt.figure(figsize=(10, 6))

plt.plot(
    df_promedio["cantidad_paradas_owner_dia"],
    df_promedio["duracion_estimada_min"],
    marker="o"
)

plt.title("Duración promedio vs cantidad de paradas (sin outliers - IQR)")
plt.xlabel("Cantidad de paradas por conductor/día")
plt.ylabel("Duración promedio (min)")
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(40, 6))

df_plot.boxplot(
    column="duracion_estimada_min",
    by="cantidad_paradas_owner_dia",
    grid=False
)

plt.title("Distribución de duración según cantidad de paradas (IQR)")
plt.suptitle("")
plt.xlabel("Cantidad de paradas por conductor/día")
plt.ylabel("Duración estimada (min)")

plt.xticks(rotation=45)
plt.ylim(0, 4)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df_hist = df_plot.copy()  # o datos_diarios_sin_outliers_iqr / df_con_comuna

df_hist["cantidad_paquetes"] = pd.to_numeric(
    df_hist["cantidad_paquetes"],
    errors="coerce"
)

df_hist = df_hist.dropna(subset=["cantidad_paquetes"])

plt.figure(figsize=(10, 6))

plt.hist(
    df_hist["cantidad_paquetes"],
    bins=30,
    edgecolor="black"
)

plt.title("Histograma de número de paquetes por dirección")
plt.xlabel("Número de paquetes por dirección")
plt.ylabel("Frecuencia")

plt.grid(axis="y", alpha=0.3)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df_hist = df_plot.copy()

df_hist["cantidad_paquetes"] = pd.to_numeric(
    df_hist["cantidad_paquetes"],
    errors="coerce"
)

df_hist = df_hist.dropna(subset=["cantidad_paquetes"])

plt.figure(figsize=(10, 6))

counts, bins, patches = plt.hist(
    df_hist["cantidad_paquetes"],
    bins=range(
        int(df_hist["cantidad_paquetes"].min()),
        int(df_hist["cantidad_paquetes"].max()) + 2
    ),
    edgecolor="black",
    align="left"
)

for count, patch in zip(counts, patches):
    if count > 0:
        x = patch.get_x() + patch.get_width() / 2
        y = patch.get_height()

        plt.text(
            x,
            y,
            f"{int(count):,}",
            ha="center",
            va="bottom",
            fontsize=9,
            rotation=90
        )

plt.title("Histograma de número de paquetes por dirección")
plt.xlabel("Número de paquetes por dirección")
plt.ylabel("Frecuencia")

plt.xticks(range(
    int(df_hist["cantidad_paquetes"].min()),
    int(df_hist["cantidad_paquetes"].max()) + 1
))

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Modelos 

In [ ]:
# ============================================================
# MODELOS | Dataset limpio con ~1% outliers eliminados
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import statsmodels.formula.api as smf

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# ============================================================
# 1. DATASET BASE
# ============================================================

df_base = datos_diarios_sin_outliers_std.copy()

target = "duracion_estimada_min"

features = [
    "Comuna",
    "dia",
    "bloque_horario",
    "cantidad_paquetes",
    "cantidad_paradas_owner_dia"
]

df_modelos = df_base[features + [target]].copy()

# Tipos numéricos
for col in [target, "cantidad_paquetes", "cantidad_paradas_owner_dia"]:
    df_modelos[col] = pd.to_numeric(df_modelos[col], errors="coerce")

# Categóricas explícitas
categorical_features = ["Comuna", "dia", "bloque_horario"]
numeric_features = ["cantidad_paquetes", "cantidad_paradas_owner_dia"]

for col in categorical_features:
    df_modelos[col] = df_modelos[col].astype("string").fillna("Sin dato")

df_modelos = df_modelos.dropna(subset=[target] + numeric_features)

X = df_modelos[features].copy()
y = df_modelos[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

df_train = X_train.copy()
df_train[target] = y_train

df_test = X_test.copy()
df_test[target] = y_test

resultados = []

def calcular_metricas(y_real, y_pred):
    y_real = np.asarray(y_real)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)

    mask = y_real != 0
    mape = np.mean(np.abs((y_real[mask] - y_pred[mask]) / y_real[mask])) * 100

    return mae, rmse, r2, mape

def guardar_resultados(nombre_modelo, y_train, pred_train, y_test, pred_test):
    for dataset, y_real, y_pred in [
        ("Train", y_train, pred_train),
        ("Test", y_test, pred_test)
    ]:
        mae, rmse, r2, mape = calcular_metricas(y_real, y_pred)

        resultados.append({
            "Modelo": nombre_modelo,
            "Dataset": dataset,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "MAPE_%": mape
        })

# Compatibilidad OneHotEncoder según versión sklearn
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", onehot, categorical_features),
        ("num", StandardScaler(), numeric_features)
    ],
    remainder="drop"
)

# ============================================================
# 2. REGRESIÓN LINEAL SIMPLE
# ============================================================

formula_lineal = """
duracion_estimada_min ~ 
C(Comuna) + 
C(dia) + 
C(bloque_horario) + 
cantidad_paquetes + 
cantidad_paradas_owner_dia
"""

modelo_lineal = smf.ols(
    formula=formula_lineal,
    data=df_train
).fit()

pred_train = modelo_lineal.predict(df_train)
pred_test = modelo_lineal.predict(df_test)

guardar_resultados(
    "Regresión Lineal Simple",
    y_train,
    pred_train,
    y_test,
    pred_test
)

# ============================================================
# 3. REGRESIÓN LINEAL CON INTERACCIONES
# Comuna x bloque_horario
# cantidad_paquetes x Comuna
# ============================================================

formula_interacciones = """
duracion_estimada_min ~ 
C(Comuna) + 
C(dia) + 
C(bloque_horario) + 
cantidad_paquetes + 
cantidad_paradas_owner_dia +
C(Comuna):C(bloque_horario) +
cantidad_paquetes:C(Comuna)
"""

modelo_lineal_interacciones = smf.ols(
    formula=formula_interacciones,
    data=df_train
).fit()

pred_train = modelo_lineal_interacciones.predict(df_train)
pred_test = modelo_lineal_interacciones.predict(df_test)

guardar_resultados(
    "Regresión Lineal con Interacciones",
    y_train,
    pred_train,
    y_test,
    pred_test
)

# ============================================================
# 4. XGBOOST
# Categóricas tratadas correctamente vía One-Hot Encoding
# ============================================================

modelo_xgb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

modelo_xgb.fit(X_train, y_train)

pred_train = modelo_xgb.predict(X_train)
pred_test = modelo_xgb.predict(X_test)

guardar_resultados(
    "XGBoost",
    y_train,
    pred_train,
    y_test,
    pred_test
)

# ============================================================
# 5. CATBOOST
# Categóricas tratadas nativamente
# ============================================================

X_train_cat = X_train.copy()
X_test_cat = X_test.copy()

for col in categorical_features:
    X_train_cat[col] = X_train_cat[col].astype(str)
    X_test_cat[col] = X_test_cat[col].astype(str)

modelo_catboost = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=0
)

modelo_catboost.fit(
    X_train_cat,
    y_train,
    cat_features=categorical_features
)

pred_train = modelo_catboost.predict(X_train_cat)
pred_test = modelo_catboost.predict(X_test_cat)

guardar_resultados(
    "CatBoost",
    y_train,
    pred_train,
    y_test,
    pred_test
)

# ============================================================
# 6. TABLA COMPARATIVA FINAL
# ============================================================

resultados_modelos = pd.DataFrame(resultados)

resultados_modelos = resultados_modelos.sort_values(
    by=["Dataset", "RMSE"],
    ascending=[True, True]
)

resultados_modelos

In [ ]:
# ============================================================
# LASSO CON VARIABLES CATEGÓRICAS COMO DUMMIES
# ============================================================

from sklearn.linear_model import LassoCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm
import numpy as np
import pandas as pd

# Compatibilidad sklearn
try:
    onehot_lasso = OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)
except TypeError:
    onehot_lasso = OneHotEncoder(handle_unknown="ignore", drop="first", sparse=False)

preprocessor_lasso = ColumnTransformer(
    transformers=[
        ("cat", onehot_lasso, categorical_features),
        ("num", StandardScaler(), numeric_features)
    ],
    remainder="drop"
)

lasso_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_lasso),
    ("model", LassoCV(
        alphas=np.logspace(-4, 2, 100),
        cv=5,
        random_state=42,
        max_iter=10000,
        n_jobs=-1
    ))
])

lasso_pipeline.fit(X_train, y_train)

# Predicciones
pred_train_lasso = lasso_pipeline.predict(X_train)
pred_test_lasso = lasso_pipeline.predict(X_test)

guardar_resultados(
    "LASSO",
    y_train,
    pred_train_lasso,
    y_test,
    pred_test_lasso
)

# ============================================================
# VARIABLES SELECCIONADAS POR LASSO
# ============================================================

feature_names_lasso = lasso_pipeline.named_steps["preprocessor"].get_feature_names_out()
coef_lasso = lasso_pipeline.named_steps["model"].coef_

lasso_importancia = pd.DataFrame({
    "variable": feature_names_lasso,
    "coeficiente_lasso": coef_lasso
})

lasso_importancia["abs_coeficiente"] = lasso_importancia["coeficiente_lasso"].abs()

lasso_importancia = lasso_importancia.sort_values(
    "abs_coeficiente",
    ascending=False
)

variables_seleccionadas_lasso = lasso_importancia[
    lasso_importancia["coeficiente_lasso"] != 0
].copy()

print("Alpha seleccionado por LASSO:", lasso_pipeline.named_steps["model"].alpha_)
print("Variables originales/dummies totales:", len(feature_names_lasso))
print("Variables seleccionadas por LASSO:", len(variables_seleccionadas_lasso))

variables_seleccionadas_lasso.head(30)

In [ ]:
# ============================================================
# OLS REFIT CON VARIABLES SELECCIONADAS POR LASSO
# ============================================================

X_train_encoded = lasso_pipeline.named_steps["preprocessor"].transform(X_train)
X_test_encoded = lasso_pipeline.named_steps["preprocessor"].transform(X_test)

selected_mask = coef_lasso != 0

X_train_lasso_selected = X_train_encoded[:, selected_mask]
X_test_lasso_selected = X_test_encoded[:, selected_mask]

selected_feature_names = feature_names_lasso[selected_mask]

X_train_lasso_selected_df = pd.DataFrame(
    X_train_lasso_selected,
    columns=selected_feature_names,
    index=X_train.index
)

X_test_lasso_selected_df = pd.DataFrame(
    X_test_lasso_selected,
    columns=selected_feature_names,
    index=X_test.index
)

X_train_lasso_selected_sm = sm.add_constant(X_train_lasso_selected_df)
X_test_lasso_selected_sm = sm.add_constant(X_test_lasso_selected_df)

modelo_ols_lasso_refit = sm.OLS(
    y_train,
    X_train_lasso_selected_sm
).fit()

pred_train_ols_lasso = modelo_ols_lasso_refit.predict(X_train_lasso_selected_sm)
pred_test_ols_lasso = modelo_ols_lasso_refit.predict(X_test_lasso_selected_sm)

guardar_resultados(
    "OLS Refit post-LASSO",
    y_train,
    pred_train_ols_lasso,
    y_test,
    pred_test_ols_lasso
)

# Tabla de significancia
tabla_significancia_lasso = pd.DataFrame({
    "variable": modelo_ols_lasso_refit.params.index,
    "coeficiente": modelo_ols_lasso_refit.params.values,
    "std_error": modelo_ols_lasso_refit.bse.values,
    "t_value": modelo_ols_lasso_refit.tvalues.values,
    "p_value": modelo_ols_lasso_refit.pvalues.values
})

tabla_significancia_lasso["significativa_5pct"] = tabla_significancia_lasso["p_value"] < 0.05

tabla_significancia_lasso = tabla_significancia_lasso.sort_values(
    "p_value",
    ascending=True
)

tabla_significancia_lasso.head(40)

In [ ]:
resultados_modelos = pd.DataFrame(resultados)

resultados_modelos.sort_values(
    by=["Dataset", "RMSE"],
    ascending=[True, True]
)

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Si estás en Colab y no tienes Optuna instalado:
# !pip install optuna

import optuna
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

In [ ]:
# ============================================================
# SPLIT PARA OPTUNA
# Train full / Test
# Luego Train / Valid dentro del Train full
# ============================================================

X_train_full, X_test_final, y_train_full, y_test_final = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train_opt, X_valid_opt, y_train_opt, y_valid_opt = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42
)

print("Train opt:", X_train_opt.shape)
print("Valid opt:", X_valid_opt.shape)
print("Train full:", X_train_full.shape)
print("Test final:", X_test_final.shape)

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluar_modelo(nombre_modelo, y_train, pred_train, y_test, pred_test):
    return pd.DataFrame([
        {
            "Modelo": nombre_modelo,
            "Dataset": "Train",
            "MAE": mean_absolute_error(y_train, pred_train),
            "RMSE": rmse(y_train, pred_train),
            "R2": r2_score(y_train, pred_train),
            "MAPE_%": np.mean(np.abs((y_train - pred_train) / y_train)) * 100
        },
        {
            "Modelo": nombre_modelo,
            "Dataset": "Test",
            "MAE": mean_absolute_error(y_test, pred_test),
            "RMSE": rmse(y_test, pred_test),
            "R2": r2_score(y_test, pred_test),
            "MAPE_%": np.mean(np.abs((y_test - pred_test) / y_test)) * 100
        }
    ])

In [ ]:
try:
    onehot_xgb = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot_xgb = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor_xgb = ColumnTransformer(
    transformers=[
        ("cat", onehot_xgb, categorical_features),
        ("num", StandardScaler(), numeric_features)
    ],
    remainder="drop"
)

In [ ]:
def objective_xgb(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "objective": "reg:squarederror",
        "random_state": 42,
        "n_jobs": -1
    }

    model = Pipeline(steps=[
        ("preprocessor", preprocessor_xgb),
        ("model", XGBRegressor(**params))
    ])

    model.fit(X_train_opt, y_train_opt)

    pred_valid = model.predict(X_valid_opt)

    return rmse(y_valid_opt, pred_valid)


study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(objective_xgb, n_trials=30)

print("Mejor RMSE validación XGBoost:", study_xgb.best_value)
print("Mejores parámetros XGBoost:")
study_xgb.best_params

In [ ]:
best_params_xgb = study_xgb.best_params.copy()

best_params_xgb.update({
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1
})

modelo_xgb_optimizado = Pipeline(steps=[
    ("preprocessor", preprocessor_xgb),
    ("model", XGBRegressor(**best_params_xgb))
])

modelo_xgb_optimizado.fit(X_train_full, y_train_full)

pred_train_xgb_opt = modelo_xgb_optimizado.predict(X_train_full)
pred_test_xgb_opt = modelo_xgb_optimizado.predict(X_test_final)

resultados_xgb_opt = evaluar_modelo(
    "XGBoost Optimizado",
    y_train_full,
    pred_train_xgb_opt,
    y_test_final,
    pred_test_xgb_opt
)

resultados_xgb_opt

In [ ]:
X_train_opt_cat = X_train_opt.copy()
X_valid_opt_cat = X_valid_opt.copy()
X_train_full_cat = X_train_full.copy()
X_test_final_cat = X_test_final.copy()

for col in categorical_features:
    X_train_opt_cat[col] = X_train_opt_cat[col].astype(str)
    X_valid_opt_cat[col] = X_valid_opt_cat[col].astype(str)
    X_train_full_cat[col] = X_train_full_cat[col].astype(str)
    X_test_final_cat[col] = X_test_final_cat[col].astype(str)


def objective_catboost(trial):

    params = {
        "iterations": trial.suggest_int("iterations", 300, 1200),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 20.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
        "loss_function": "RMSE",
        "random_seed": 42,
        "verbose": 0
    }

    model = CatBoostRegressor(**params)

    model.fit(
        X_train_opt_cat,
        y_train_opt,
        cat_features=categorical_features,
        eval_set=(X_valid_opt_cat, y_valid_opt),
        verbose=0
    ) 

    pred_valid = model.predict(X_valid_opt_cat)

    return rmse(y_valid_opt, pred_valid)


study_catboost = optuna.create_study(direction="minimize")
study_catboost.optimize(objective_catboost, n_trials=30)

print("Mejor RMSE validación CatBoost:", study_catboost.best_value)
print("Mejores parámetros CatBoost:")
study_catboost.best_params

In [ ]:
best_params_catboost = study_catboost.best_params.copy()

best_params_catboost.update({
    "loss_function": "RMSE",
    "random_seed": 42,
    "verbose": 0
})

modelo_catboost_optimizado = CatBoostRegressor(**best_params_catboost)

modelo_catboost_optimizado.fit(
    X_train_full_cat,
    y_train_full,
    cat_features=categorical_features,
    verbose=0
)

pred_train_cat_opt = modelo_catboost_optimizado.predict(X_train_full_cat)
pred_test_cat_opt = modelo_catboost_optimizado.predict(X_test_final_cat)

resultados_catboost_opt = evaluar_modelo(
    "CatBoost Optimizado",
    y_train_full,
    pred_train_cat_opt,
    y_test_final,
    pred_test_cat_opt
)

resultados_catboost_opt

In [ ]:
resultados_optimizados = pd.concat(
    [
        resultados_modelos,
        resultados_xgb_opt,
        resultados_catboost_opt
    ],
    ignore_index=True
)

resultados_optimizados.sort_values(
    by=["Dataset", "RMSE"],
    ascending=[True, True]
)

## Generamos tabla con validation

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42
)

In [ ]:
df_train = X_train.copy()
df_train[target] = y_train

df_test = X_test.copy()
df_test[target] = y_test

df_valid = X_valid.copy()
df_valid[target] = y_valid

In [ ]:
def guardar_resultados(
    nombre_modelo,
    y_train, pred_train,
    y_valid, pred_valid,
    y_test, pred_test
):
    for dataset, y_real, y_pred in [
        ("Train", y_train, pred_train),
        ("Validation", y_valid, pred_valid),
        ("Test", y_test, pred_test)
    ]:
        mae, rmse, r2, mape = calcular_metricas(y_real, y_pred)

        resultados.append({
            "Modelo": nombre_modelo,
            "Dataset": dataset,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "MAPE_%": mape
        })

In [ ]:
pred_train = modelo_lineal.predict(df_train)
pred_valid = modelo_lineal.predict(df_valid)
pred_test = modelo_lineal.predict(df_test)

guardar_resultados(
    "Regresión Lineal Simple",
    y_train,
    pred_train,
    y_valid,
    pred_valid,
    y_test,
    pred_test
)

In [ ]:
pred_train = modelo_xgb.predict(X_train)
pred_valid = modelo_xgb.predict(X_valid)
pred_test = modelo_xgb.predict(X_test)

guardar_resultados(
    "XGBoost",
    y_train,
    pred_train,
    y_valid,
    pred_valid,
    y_test,
    pred_test
)

In [ ]:
X_valid_cat = X_valid.copy()

for col in categorical_features:
    X_valid_cat[col] = X_valid_cat[col].astype(str)

pred_train = modelo_catboost.predict(X_train_cat)
pred_valid = modelo_catboost.predict(X_valid_cat)
pred_test = modelo_catboost.predict(X_test_cat)

guardar_resultados(
    "CatBoost",
    y_train,
    pred_train,
    y_valid,
    pred_valid,
    y_test,
    pred_test
)

## Veamos cuales son las variables relevantes por modelo

In [ ]:
modelo_lineal.summary()

In [ ]:
tabla_ols = pd.DataFrame({
    "variable": modelo_lineal.params.index,
    "coeficiente": modelo_lineal.params.values,
    "std_error": modelo_lineal.bse.values,
    "t_value": modelo_lineal.tvalues.values,
    "p_value": modelo_lineal.pvalues.values
})

tabla_ols["significativa_5pct"] = tabla_ols["p_value"] < 0.05

tabla_ols = tabla_ols.sort_values("p_value")

tabla_ols.head(40)

In [ ]:
variables_seleccionadas_lasso.head(40)

In [ ]:
modelo_xgb_optimizado.named_steps["model"]

In [ ]:
xgb_model = modelo_xgb_optimizado.named_steps["model"]

feature_names_xgb = modelo_xgb_optimizado.named_steps[
    "preprocessor"
].get_feature_names_out()

xgb_importance = pd.DataFrame({
    "variable": feature_names_xgb,
    "importance": xgb_model.feature_importances_
})

xgb_importance = xgb_importance.sort_values(
    "importance",
    ascending=False
)

xgb_importance.head(30)

In [ ]:
cat_importance = pd.DataFrame({
    "variable": modelo_catboost_optimizado.feature_names_,
    "importance": modelo_catboost_optimizado.get_feature_importance()
})

cat_importance = cat_importance.sort_values(
    "importance",
    ascending=False
)

cat_importance

In [ ]:
%pip install shap

In [ ]:
# ============================================================
# SHAP PARA CATBOOST OPTIMIZADO
# ============================================================

# Si estás en Colab y no tienes shap:
# !pip install shap

import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Muestra para no hacer SHAP sobre todo el dataset
X_shap = X_test_final_cat.copy()

# Opcional: limitar muestra para que corra más rápido
X_shap_sample = X_shap.sample(
    n=min(3000, len(X_shap)),
    random_state=42
)

# Crear explainer
explainer_cat = shap.TreeExplainer(modelo_catboost_optimizado)

# Calcular valores SHAP
shap_values_cat = explainer_cat.shap_values(X_shap_sample)

# ============================================================
# 1. IMPORTANCIA GLOBAL PROMEDIO
# ============================================================

shap_importance_cat = pd.DataFrame({
    "variable": X_shap_sample.columns,
    "mean_abs_shap": np.abs(shap_values_cat).mean(axis=0)
})

shap_importance_cat = shap_importance_cat.sort_values(
    "mean_abs_shap",
    ascending=False
)

shap_importance_cat

In [ ]:
# ============================================================
# 2. GRÁFICO DE IMPORTANCIA GLOBAL
# ============================================================

shap.summary_plot(
    shap_values_cat,
    X_shap_sample,
    plot_type="bar"
)

In [ ]:
# ============================================================
# 3. SUMMARY PLOT
# Muestra importancia + dirección del efecto
# ============================================================

shap.summary_plot(
    shap_values_cat,
    X_shap_sample
)

In [ ]:
shap.dependence_plot("Comuna", shap_values_cat, X_shap_sample)

In [ ]:
# ============================================================
# DRILL-DOWN SHAP POR COMUNA
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# índice de la variable comuna dentro del array SHAP
idx_comuna = list(X_shap_sample.columns).index("Comuna")

# construir dataframe
drilldown_comuna = pd.DataFrame({
    "Comuna": X_shap_sample["Comuna"].values,
    "shap_comuna": shap_values_cat[:, idx_comuna]
})

# agregación por comuna
resumen_comuna = (
    drilldown_comuna
    .groupby("Comuna")
    .agg(
        impacto_promedio=("shap_comuna", "mean"),
        impacto_abs_promedio=("shap_comuna", lambda x: np.mean(np.abs(x))),
        n_obs=("shap_comuna", "count")
    )
    .reset_index()
)

# filtrar comunas con pocas observaciones para evitar ruido
resumen_comuna = resumen_comuna[
    resumen_comuna["n_obs"] >= 30
]

# ordenar
resumen_comuna = resumen_comuna.sort_values(
    "impacto_promedio",
    ascending=False
)

resumen_comuna

In [ ]:
top_aumentan = resumen_comuna.head(15)

plt.figure(figsize=(10,6))
plt.barh(
    top_aumentan["Comuna"],
    top_aumentan["impacto_promedio"]
)

plt.xlabel("Impacto SHAP promedio (minutos)")
plt.ylabel("Comuna")
plt.title("Top comunas que aumentan tiempo de servicio")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
top_reducen = resumen_comuna.tail(15)

plt.figure(figsize=(10,6))
plt.barh(
    top_reducen["Comuna"],
    top_reducen["impacto_promedio"]
)

plt.xlabel("Impacto SHAP promedio (minutos)")
plt.ylabel("Comuna")
plt.title("Top comunas que reducen tiempo de servicio")
plt.show()

In [ ]:
# ============================================================
# DRILL-DOWN SHAP POR BLOQUE HORARIO
# ============================================================

idx_bloque = list(X_shap_sample.columns).index("bloque_horario")

drilldown_bloque = pd.DataFrame({
    "bloque_horario": X_shap_sample["bloque_horario"].values,
    "shap_bloque": shap_values_cat[:, idx_bloque]
})

resumen_bloque = (
    drilldown_bloque
    .groupby("bloque_horario")
    .agg(
        impacto_promedio=("shap_bloque", "mean"),
        impacto_abs_promedio=("shap_bloque", lambda x: np.mean(np.abs(x))),
        n_obs=("shap_bloque", "count")
    )
    .reset_index()
)

# ordenar correctamente numéricamente
resumen_bloque["bloque_horario"] = pd.to_numeric(
    resumen_bloque["bloque_horario"],
    errors="coerce"
)

resumen_bloque = resumen_bloque.sort_values("bloque_horario")

resumen_bloque

In [ ]:
plt.figure(figsize=(12,6))

plt.bar(
    resumen_bloque["bloque_horario"].astype(str),
    resumen_bloque["impacto_promedio"]
)

plt.axhline(0, linestyle="--")

plt.xlabel("Bloque horario")
plt.ylabel("Impacto SHAP promedio")
plt.title("Impacto promedio del bloque horario sobre tiempo de servicio")

plt.xticks(rotation=45)
plt.show()

In [ ]:
shap.dependence_plot("cantidad_paquetes", shap_values_cat, X_shap_sample)

In [ ]:
shap.dependence_plot("cantidad_paradas_owner_dia", shap_values_cat, X_shap_sample)

## Cuanto tiempo viajamos y cuando tiempo servimos?

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# COPIA SEGURA
# ============================================================

df_rutas = df_con_comuna.copy()

# ============================================================
# FORMATO DATETIME
# ============================================================

df_rutas["created_inicio"] = pd.to_datetime(
    df_rutas["created_inicio"],
    errors="coerce"
)

# duración numérica
df_rutas["duracion_estimada_min"] = pd.to_numeric(
    df_rutas["duracion_estimada_min"],
    errors="coerce"
)

# ============================================================
# CREAR TIMESTAMP DE FIN DE ENTREGA
# ============================================================

df_rutas["created_fin_entrega"] = (
    df_rutas["created_inicio"] +
    pd.to_timedelta(df_rutas["duracion_estimada_min"], unit="m")
)

# ============================================================
# AGRUPACIÓN POR USUARIO Y DÍA
# ============================================================

resumen_rutas = (
    df_rutas
    .groupby(["owner_id", "dia"])
    .agg(
        inicio_ruta=("created_inicio", "min"),
        fin_ruta=("created_fin_entrega", "max"),
        tiempo_entregando_min=("duracion_estimada_min", "sum"),
        cantidad_paradas=("parada_agrupada_detectada_id", "count")
    )
    .reset_index()
)

# ============================================================
# TIEMPO TOTAL DE RUTA
# ============================================================

resumen_rutas["tiempo_total_ruta_min"] = (
    (
        resumen_rutas["fin_ruta"] -
        resumen_rutas["inicio_ruta"]
    )
    .dt.total_seconds() / 60
)

# ============================================================
# TIEMPO MANEJANDO
# ============================================================

resumen_rutas["tiempo_manejando_min"] = (
    resumen_rutas["tiempo_total_ruta_min"] -
    resumen_rutas["tiempo_entregando_min"]
)

# Evitar negativos raros
resumen_rutas["tiempo_manejando_min"] = (
    resumen_rutas["tiempo_manejando_min"]
    .clip(lower=0)
)

# ============================================================
# PORCENTAJES
# ============================================================

resumen_rutas["pct_entregando"] = (
    resumen_rutas["tiempo_entregando_min"] /
    resumen_rutas["tiempo_total_ruta_min"]
) * 100

resumen_rutas["pct_manejando"] = (
    resumen_rutas["tiempo_manejando_min"] /
    resumen_rutas["tiempo_total_ruta_min"]
) * 100

# ============================================================
# REDONDEO
# ============================================================

columnas_redondear = [
    "tiempo_entregando_min",
    "tiempo_total_ruta_min",
    "tiempo_manejando_min",
    "pct_entregando",
    "pct_manejando"
]

resumen_rutas[columnas_redondear] = (
    resumen_rutas[columnas_redondear]
    .round(2)
)

# ============================================================
# RESULTADO
# ============================================================

resumen_rutas.head()

In [ ]:
# ============================================================
# TABLA RESUMEN POR DÍA
# ============================================================

tabla_por_dia = (
    resumen_rutas
    .groupby("dia")
    .agg(
        rutas=("owner_id", "count"),

        tiempo_total_ruta_min_promedio=(
            "tiempo_total_ruta_min",
            "mean"
        ),

        tiempo_entregando_min_promedio=(
            "tiempo_entregando_min",
            "mean"
        ),

        tiempo_manejando_min_promedio=(
            "tiempo_manejando_min",
            "mean"
        ),

        pct_entregando_promedio=(
            "pct_entregando",
            "mean"
        ),

        pct_manejando_promedio=(
            "pct_manejando",
            "mean"
        )
    )
    .reset_index()
)

# ============================================================
# REDONDEAR
# ============================================================

columnas_redondear = [
    "tiempo_total_ruta_min_promedio",
    "tiempo_entregando_min_promedio",
    "tiempo_manejando_min_promedio",
    "pct_entregando_promedio",
    "pct_manejando_promedio"
]

tabla_por_dia[columnas_redondear] = (
    tabla_por_dia[columnas_redondear]
    .round(2)
)

# ============================================================
# ORDEN SEMANAL
# ============================================================

orden_dias = [
    "lunes",
    "martes",
    "miércoles",
    "jueves",
    "viernes",
    "sábado",
    "domingo"
]

tabla_por_dia["dia"] = pd.Categorical(
    tabla_por_dia["dia"],
    categories=orden_dias,
    ordered=True
)

tabla_por_dia = tabla_por_dia.sort_values("dia")

# ============================================================
# RESULTADO
# ============================================================


tabla_por_dia

## Modelos con datos SOLO de la RM

In [ ]:
# ============================================================
# SECCIÓN ADICIONAL | FILTRO SOLO COMUNAS RM
# ============================================================

import unicodedata
import pandas as pd

def normalizar_texto(x):
    if pd.isna(x):
        return x
    x = str(x).strip().upper()
    x = ''.join(
        c for c in unicodedata.normalize('NFD', x)
        if unicodedata.category(c) != 'Mn'
    )
    return x

comunas_rm = [
    "CERRILLOS", "CERRO NAVIA", "CONCHALI", "EL BOSQUE", "ESTACION CENTRAL",
    "HUECHURABA", "INDEPENDENCIA", "LA CISTERNA", "LA FLORIDA", "LA GRANJA",
    "LA PINTANA", "LA REINA", "LAS CONDES", "LO BARNECHEA", "LO ESPEJO",
    "LO PRADO", "MACUL", "MAIPU", "NUNOA", "PEDRO AGUIRRE CERDA",
    "PENALOLEN", "PROVIDENCIA", "PUDAHUEL", "QUILICURA", "QUINTA NORMAL",
    "RECOLETA", "RENCA", "SAN JOAQUIN", "SAN MIGUEL", "SAN RAMON",
    "SANTIAGO", "VITACURA",

    "PUENTE ALTO", "PIRQUE", "SAN JOSE DE MAIPO",
    "COLINA", "LAMPA", "TILTIL",
    "SAN BERNARDO", "BUIN", "CALERA DE TANGO", "PAINE",
    "MELIPILLA", "ALHUE", "CURACAVI", "MARIA PINTO", "SAN PEDRO",
    "TALAGANTE", "EL MONTE", "ISLA DE MAIPO", "PADRE HURTADO", "PENAFLOR"
]

comunas_rm_norm = set(comunas_rm)

df_base_rm = datos_diarios_sin_outliers_std.copy()

df_base_rm["Comuna_norm"] = df_base_rm["Comuna"].apply(normalizar_texto)

df_base_rm = df_base_rm[
    df_base_rm["Comuna_norm"].isin(comunas_rm_norm)
].copy()

df_base_rm = df_base_rm.drop(columns=["Comuna_norm"])

print("Filas base original:", len(datos_diarios_sin_outliers_std))
print("Filas solo RM:", len(df_base_rm))
print("Comunas RM detectadas:", df_base_rm["Comuna"].nunique())

df_base_rm["Comuna"].value_counts().sort_index()

In [ ]:
# ============================================================
# MODELOS | SOLO REGIÓN METROPOLITANA | TRAIN / VALIDATION / TEST
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import statsmodels.formula.api as smf

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# ============================================================
# 1. DATASET BASE RM
# ============================================================

df_base = df_base_rm.copy()

target = "duracion_estimada_min"

features = [
    "Comuna",
    "dia",
    "bloque_horario",
    "cantidad_paquetes",
    "cantidad_paradas_owner_dia"
]

df_modelos_rm = df_base[features + [target]].copy()

for col in [target, "cantidad_paquetes", "cantidad_paradas_owner_dia"]:
    df_modelos_rm[col] = pd.to_numeric(df_modelos_rm[col], errors="coerce")

categorical_features = ["Comuna", "dia", "bloque_horario"]
numeric_features = ["cantidad_paquetes", "cantidad_paradas_owner_dia"]

for col in categorical_features:
    df_modelos_rm[col] = df_modelos_rm[col].astype("string").fillna("Sin dato")

df_modelos_rm = df_modelos_rm.dropna(subset=[target] + numeric_features)

X = df_modelos_rm[features].copy()
y = df_modelos_rm[target].copy()

# Split final:
# Train full = 80%
# Test = 20%
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Split interno:
# Train = 64%
# Validation = 16%
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42
)

df_train = X_train.copy()
df_train[target] = y_train

df_valid = X_valid.copy()
df_valid[target] = y_valid

df_test = X_test.copy()
df_test[target] = y_test

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

resultados_rm = []

def calcular_metricas(y_real, y_pred):
    y_real = np.asarray(y_real)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)

    mask = y_real != 0
    mape = np.mean(np.abs((y_real[mask] - y_pred[mask]) / y_real[mask])) * 100

    return mae, rmse, r2, mape

def guardar_resultados(nombre_modelo, y_train, pred_train, y_valid, pred_valid, y_test, pred_test):
    for dataset, y_real, y_pred in [
        ("Train", y_train, pred_train),
        ("Validation", y_valid, pred_valid),
        ("Test", y_test, pred_test)
    ]:
        mae, rmse, r2, mape = calcular_metricas(y_real, y_pred)

        resultados_rm.append({
            "Modelo": nombre_modelo,
            "Dataset": dataset,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "MAPE_%": mape
        })

try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", onehot, categorical_features),
        ("num", StandardScaler(), numeric_features)
    ],
    remainder="drop"
)

# ============================================================
# 2. REGRESIÓN LINEAL SIMPLE
# ============================================================

formula_lineal = """
duracion_estimada_min ~ 
C(Comuna) + 
C(dia) + 
C(bloque_horario) + 
cantidad_paquetes + 
cantidad_paradas_owner_dia
"""

modelo_lineal_rm = smf.ols(
    formula=formula_lineal,
    data=df_train
).fit()

pred_train = modelo_lineal_rm.predict(df_train)
pred_valid = modelo_lineal_rm.predict(df_valid)
pred_test = modelo_lineal_rm.predict(df_test)

guardar_resultados(
    "Regresión Lineal Simple",
    y_train, pred_train,
    y_valid, pred_valid,
    y_test, pred_test
)

# ============================================================
# 3. REGRESIÓN LINEAL CON INTERACCIONES
# ============================================================

formula_interacciones = """
duracion_estimada_min ~ 
C(Comuna) + 
C(dia) + 
C(bloque_horario) + 
cantidad_paquetes + 
cantidad_paradas_owner_dia +
C(Comuna):C(bloque_horario) +
cantidad_paquetes:C(Comuna)
"""

modelo_lineal_interacciones_rm = smf.ols(
    formula=formula_interacciones,
    data=df_train
).fit()

pred_train = modelo_lineal_interacciones_rm.predict(df_train)
pred_valid = modelo_lineal_interacciones_rm.predict(df_valid)
pred_test = modelo_lineal_interacciones_rm.predict(df_test)

guardar_resultados(
    "Regresión Lineal con Interacciones",
    y_train, pred_train,
    y_valid, pred_valid,
    y_test, pred_test
)

# ============================================================
# 4. XGBOOST
# ============================================================

modelo_xgb_rm = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

modelo_xgb_rm.fit(X_train, y_train)

pred_train = modelo_xgb_rm.predict(X_train)
pred_valid = modelo_xgb_rm.predict(X_valid)
pred_test = modelo_xgb_rm.predict(X_test)

guardar_resultados(
    "XGBoost",
    y_train, pred_train,
    y_valid, pred_valid,
    y_test, pred_test
)

# ============================================================
# 5. CATBOOST
# ============================================================

X_train_cat = X_train.copy()
X_valid_cat = X_valid.copy()
X_test_cat = X_test.copy()

for col in categorical_features:
    X_train_cat[col] = X_train_cat[col].astype(str)
    X_valid_cat[col] = X_valid_cat[col].astype(str)
    X_test_cat[col] = X_test_cat[col].astype(str)

modelo_catboost_rm = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=0
)

modelo_catboost_rm.fit(
    X_train_cat,
    y_train,
    cat_features=categorical_features
)

pred_train = modelo_catboost_rm.predict(X_train_cat)
pred_valid = modelo_catboost_rm.predict(X_valid_cat)
pred_test = modelo_catboost_rm.predict(X_test_cat)

guardar_resultados(
    "CatBoost",
    y_train, pred_train,
    y_valid, pred_valid,
    y_test, pred_test
)

# ============================================================
# 6. TABLA COMPARATIVA RM
# ============================================================

resultados_modelos_rm = pd.DataFrame(resultados_rm)

resultados_modelos_rm = resultados_modelos_rm.sort_values(
    by=["Dataset", "RMSE"],
    ascending=[True, True]
)

resultados_modelos_rm

In [ ]:

# 7. LASSO | SOLO REGIÓN METROPOLITANA


from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline

modelo_lasso_rm = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LassoCV(
        alphas=np.logspace(-4, 2, 100),
        cv=5,
        random_state=42,
        max_iter=10000,
        n_jobs=-1
    ))
])

modelo_lasso_rm.fit(X_train, y_train)

pred_train = modelo_lasso_rm.predict(X_train)
pred_valid = modelo_lasso_rm.predict(X_valid)
pred_test = modelo_lasso_rm.predict(X_test)

guardar_resultados(
    "LASSO",
    y_train, pred_train,
    y_valid, pred_valid,
    y_test, pred_test
)

print("Alpha óptimo LASSO RM:", modelo_lasso_rm.named_steps["model"].alpha_)

In [ ]:
resultados_modelos_rm = pd.DataFrame(resultados_rm)

resultados_modelos_rm = resultados_modelos_rm.sort_values(
    by=["Modelo", "Dataset"]
)

resultados_modelos_rm

In [ ]:
# ============================================================
# 8. OLS REFIT POST-LASSO | SOLO REGIÓN METROPOLITANA
# ============================================================

import statsmodels.api as sm

# Recuperar nombres de variables después del preprocesamiento
feature_names_lasso_rm = modelo_lasso_rm.named_steps["preprocessor"].get_feature_names_out()

coef_lasso_rm = modelo_lasso_rm.named_steps["model"].coef_

variables_seleccionadas_rm = pd.DataFrame({
    "variable": feature_names_lasso_rm,
    "coeficiente_lasso": coef_lasso_rm
})

variables_seleccionadas_rm = variables_seleccionadas_rm[
    variables_seleccionadas_rm["coeficiente_lasso"] != 0
].copy()

print("Número de variables seleccionadas por LASSO RM:", len(variables_seleccionadas_rm))

display(
    variables_seleccionadas_rm
    .sort_values("coeficiente_lasso", key=abs, ascending=False)
    .head(30)
)

# Transformar datasets usando el mismo preprocessor ya entrenado
X_train_lasso_trans = modelo_lasso_rm.named_steps["preprocessor"].transform(X_train)
X_valid_lasso_trans = modelo_lasso_rm.named_steps["preprocessor"].transform(X_valid)
X_test_lasso_trans = modelo_lasso_rm.named_steps["preprocessor"].transform(X_test)

# Convertir a DataFrame para seleccionar columnas por nombre
X_train_lasso_df = pd.DataFrame(
    X_train_lasso_trans,
    columns=feature_names_lasso_rm,
    index=X_train.index
)

X_valid_lasso_df = pd.DataFrame(
    X_valid_lasso_trans,
    columns=feature_names_lasso_rm,
    index=X_valid.index
)

X_test_lasso_df = pd.DataFrame(
    X_test_lasso_trans,
    columns=feature_names_lasso_rm,
    index=X_test.index
)

selected_features_rm = variables_seleccionadas_rm["variable"].tolist()

X_train_selected_rm = X_train_lasso_df[selected_features_rm]
X_valid_selected_rm = X_valid_lasso_df[selected_features_rm]
X_test_selected_rm = X_test_lasso_df[selected_features_rm]

# Agregar constante para OLS
X_train_selected_const_rm = sm.add_constant(X_train_selected_rm, has_constant="add")
X_valid_selected_const_rm = sm.add_constant(X_valid_selected_rm, has_constant="add")
X_test_selected_const_rm = sm.add_constant(X_test_selected_rm, has_constant="add")

modelo_ols_post_lasso_rm = sm.OLS(
    y_train,
    X_train_selected_const_rm
).fit()

pred_train = modelo_ols_post_lasso_rm.predict(X_train_selected_const_rm)
pred_valid = modelo_ols_post_lasso_rm.predict(X_valid_selected_const_rm)
pred_test = modelo_ols_post_lasso_rm.predict(X_test_selected_const_rm)

guardar_resultados(
    "OLS refit post-LASSO",
    y_train, pred_train,
    y_valid, pred_valid,
    y_test, pred_test
)

resultados_modelos_rm = pd.DataFrame(resultados_rm)

resultados_modelos_rm = resultados_modelos_rm.sort_values(
    by=["Modelo", "Dataset"]
)

resultados_modelos_rm

In [ ]:
# ============================================================
# 9. XGBOOST OPTIMIZADO CON OPTUNA | SOLO REGIÓN METROPOLITANA
# ============================================================

import optuna
from sklearn.base import clone

def objective_xgb_rm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "objective": "reg:squarederror",
        "random_state": 42,
        "n_jobs": -1
    }

    modelo = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", XGBRegressor(**params))
    ])

    modelo.fit(X_train, y_train)

    pred_valid = modelo.predict(X_valid)

    rmse_valid = np.sqrt(mean_squared_error(y_valid, pred_valid))

    return rmse_valid


study_xgb_rm = optuna.create_study(
    direction="minimize",
    study_name="xgboost_rm"
)

study_xgb_rm.optimize(
    objective_xgb_rm,
    n_trials=50,
    show_progress_bar=True
)

print("Mejor RMSE validación XGBoost RM:", study_xgb_rm.best_value)
print("Mejores parámetros XGBoost RM:")
print(study_xgb_rm.best_params)

In [ ]:
# ============================================================
# 10. ENTRENAMIENTO FINAL XGBOOST OPTIMIZADO | RM
# ============================================================

best_params_xgb_rm = study_xgb_rm.best_params.copy()

best_params_xgb_rm.update({
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1
})

modelo_xgb_optimizado_rm = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", XGBRegressor(**best_params_xgb_rm))
])

modelo_xgb_optimizado_rm.fit(X_train, y_train)

pred_train = modelo_xgb_optimizado_rm.predict(X_train)
pred_valid = modelo_xgb_optimizado_rm.predict(X_valid)
pred_test = modelo_xgb_optimizado_rm.predict(X_test)

guardar_resultados(
    "XGBoost Optimizado",
    y_train, pred_train,
    y_valid, pred_valid,
    y_test, pred_test
)

resultados_modelos_rm = pd.DataFrame(resultados_rm)

resultados_modelos_rm = resultados_modelos_rm.sort_values(
    by=["Modelo", "Dataset"]
)

resultados_modelos_rm

In [ ]:
# 11. CATBOOST OPTIMIZADO CON OPTUNA | SOLO REGIÓN METROPOLITANA


import optuna

def objective_catboost_rm(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 300, 1500),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 20.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 10.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 10.0),
        "loss_function": "RMSE",
        "random_seed": 42,
        "verbose": 0
    }

    modelo = CatBoostRegressor(**params)

    modelo.fit(
        X_train_cat,
        y_train,
        cat_features=categorical_features,
        eval_set=(X_valid_cat, y_valid),
        use_best_model=True,
        verbose=0
    )

    pred_valid = modelo.predict(X_valid_cat)

    rmse_valid = np.sqrt(mean_squared_error(y_valid, pred_valid))

    return rmse_valid


study_catboost_rm = optuna.create_study(
    direction="minimize",
    study_name="catboost_rm"
)

study_catboost_rm.optimize(
    objective_catboost_rm,
    n_trials=10,
    show_progress_bar=True
)

print("Mejor RMSE validación CatBoost RM:", study_catboost_rm.best_value)
print("Mejores parámetros CatBoost RM:")
print(study_catboost_rm.best_params)

In [ ]:
# ============================================================
# 12. ENTRENAMIENTO FINAL CATBOOST OPTIMIZADO | RM
# ============================================================

best_params_catboost_rm = study_catboost_rm.best_params.copy()

best_params_catboost_rm.update({
    "loss_function": "RMSE",
    "random_seed": 42,
    "verbose": 0
})

modelo_catboost_optimizado_rm = CatBoostRegressor(**best_params_catboost_rm)

modelo_catboost_optimizado_rm.fit(
    X_train_cat,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_valid_cat, y_valid),
    use_best_model=True,
    verbose=0
)

pred_train = modelo_catboost_optimizado_rm.predict(X_train_cat)
pred_valid = modelo_catboost_optimizado_rm.predict(X_valid_cat)
pred_test = modelo_catboost_optimizado_rm.predict(X_test_cat)

guardar_resultados(
    "CatBoost Optimizado",
    y_train, pred_train,
    y_valid, pred_valid,
    y_test, pred_test
)

resultados_modelos_rm = pd.DataFrame(resultados_rm)

resultados_modelos_rm = resultados_modelos_rm.sort_values(
    by=["Modelo", "Dataset"]
)

resultados_modelos_rm

## Miremos variables relevantes

In [ ]:
modelo_catboost_mejor_rm = modelo_catboost_optimizado_rm
X_cat_mejor = X_train_cat

In [ ]:
# ============================================================
# IMPORTANCIA DE VARIABLES | CATBOOST RM
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

importancias_catboost_rm = pd.DataFrame({
    "variable": X_cat_mejor.columns,
    "importancia": modelo_catboost_mejor_rm.get_feature_importance()
})

importancias_catboost_rm = importancias_catboost_rm.sort_values(
    "importancia",
    ascending=False
)

display(importancias_catboost_rm)

plt.figure(figsize=(10, 6))
plt.barh(
    importancias_catboost_rm["variable"],
    importancias_catboost_rm["importancia"]
)
plt.gca().invert_yaxis()
plt.title("Importancia de variables - CatBoost RM")
plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.show()

In [ ]:
# ============================================================
# SHAP VALUES | CATBOOST RM | VARIABLES CATEGÓRICAS DESAGREGADAS
# ============================================================

import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Muestra para evitar que SHAP demore demasiado
X_shap_cat = X_cat_mejor.copy()

for col in categorical_features:
    X_shap_cat[col] = X_shap_cat[col].astype(str)

if len(X_shap_cat) > 5000:
    X_shap_cat_sample = X_shap_cat.sample(5000, random_state=42)
else:
    X_shap_cat_sample = X_shap_cat.copy()

explainer_catboost_rm = shap.TreeExplainer(modelo_catboost_mejor_rm)

shap_values_catboost_rm = explainer_catboost_rm.shap_values(X_shap_cat_sample)

shap_catboost_df_rm = pd.DataFrame(
    shap_values_catboost_rm,
    columns=X_shap_cat_sample.columns,
    index=X_shap_cat_sample.index
)

shap_catboost_long_rm = []

for variable in ["bloque_horario", "Comuna", "dia"]:
    temp = pd.DataFrame({
        "variable": variable,
        "categoria": X_shap_cat_sample[variable],
        "shap_value": shap_catboost_df_rm[variable]
    })

    temp["abs_shap_value"] = temp["shap_value"].abs()

    shap_catboost_long_rm.append(temp)

shap_catboost_long_rm = pd.concat(shap_catboost_long_rm, ignore_index=True)

In [ ]:
# ============================================================
# APORTE PROMEDIO DESAGREGADO POR CATEGORÍA
# ============================================================

aporte_categorias_catboost_rm = (
    shap_catboost_long_rm
    .groupby(["variable", "categoria"], as_index=False)
    .agg(
        aporte_promedio=("shap_value", "mean"),
        aporte_abs_promedio=("abs_shap_value", "mean"),
        n_observaciones=("shap_value", "size")
    )
    .sort_values(["variable", "aporte_abs_promedio"], ascending=[True, False])
)

display(aporte_categorias_catboost_rm)

In [ ]:
aporte_bloque_horario_rm = aporte_categorias_catboost_rm[
    aporte_categorias_catboost_rm["variable"] == "bloque_horario"
].sort_values("aporte_abs_promedio", ascending=False)

aporte_comuna_rm = aporte_categorias_catboost_rm[
    aporte_categorias_catboost_rm["variable"] == "Comuna"
].sort_values("aporte_abs_promedio", ascending=False)

aporte_dia_rm = aporte_categorias_catboost_rm[
    aporte_categorias_catboost_rm["variable"] == "dia"
].sort_values("aporte_abs_promedio", ascending=False)

display(aporte_bloque_horario_rm)
display(aporte_comuna_rm)
display(aporte_dia_rm)

In [ ]:
def graficar_aporte_catboost(df, titulo, top_n=20):
    df_plot = df.head(top_n).sort_values("aporte_abs_promedio", ascending=True)

    plt.figure(figsize=(10, 6))
    plt.barh(df_plot["categoria"].astype(str), df_plot["aporte_abs_promedio"])
    plt.title(titulo)
    plt.xlabel("Aporte absoluto promedio SHAP")
    plt.ylabel("Categoría")
    plt.show()

graficar_aporte_catboost(
    aporte_bloque_horario_rm,
    "Aporte desagregado por bloque horario - CatBoost RM",
    top_n=24
)

graficar_aporte_catboost(
    aporte_comuna_rm,
    "Top comunas con mayor aporte - CatBoost RM",
    top_n=20
)

graficar_aporte_catboost(
    aporte_dia_rm,
    "Aporte desagregado por día - CatBoost RM",
    top_n=7
)

In [ ]:
def graficar_aporte_promedio_catboost(df, titulo, top_n=None):
    
    df_plot = df.copy()

    if top_n is not None:
        df_plot = (
            df_plot
            .sort_values("aporte_promedio")
        )

        negativos = df_plot.head(top_n // 2)
        positivos = df_plot.tail(top_n // 2)

        df_plot = pd.concat([negativos, positivos])

    df_plot = df_plot.sort_values("aporte_promedio")

    plt.figure(figsize=(10, 6))
    plt.barh(
        df_plot["categoria"].astype(str),
        df_plot["aporte_promedio"]
    )

    plt.axvline(0, linestyle="--")
    plt.title(titulo)
    plt.xlabel("Aporte promedio SHAP (minutos)")
    plt.ylabel("Categoría")
    plt.show()

In [ ]:
graficar_aporte_promedio_catboost(
    aporte_bloque_horario_rm,
    "Aporte promedio por bloque horario - CatBoost RM",
    top_n=24
)

In [ ]:
graficar_aporte_promedio_catboost(
    aporte_dia_rm,
    "Aporte promedio por día - CatBoost RM",
    top_n=7
)

In [ ]:
graficar_aporte_promedio_catboost(
    aporte_comuna_rm,
    "Comunas que más aumentan / reducen duración - CatBoost RM",
    top_n=40
)

In [ ]:
# ============================================================
# SELECCIONAR MEJOR REGRESIÓN LINEAL | RM
# ============================================================

modelos_lineales_rm = [
    "Regresión Lineal Simple",
    "Regresión Lineal con Interacciones",
    "LASSO",
    "OLS refit post-LASSO"
]

ranking_lineales_rm = resultados_modelos_rm[
    (resultados_modelos_rm["Modelo"].isin(modelos_lineales_rm)) &
    (resultados_modelos_rm["Dataset"] == "Validation")
].sort_values("RMSE")

display(ranking_lineales_rm)

mejor_lineal_rm = ranking_lineales_rm.iloc[0]["Modelo"]

print("Mejor modelo lineal RM según Validation RMSE:", mejor_lineal_rm)

In [ ]:
# ============================================================
# VARIABLES RELEVANTES | MEJOR REGRESIÓN LINEAL RM
# ============================================================

if mejor_lineal_rm == "Regresión Lineal Simple":
    modelo_lineal_mejor_rm = modelo_lineal_rm
    coef_lineal_rm = modelo_lineal_mejor_rm.params.reset_index()
    coef_lineal_rm.columns = ["variable", "coeficiente"]
    coef_lineal_rm["p_value"] = modelo_lineal_mejor_rm.pvalues.values

elif mejor_lineal_rm == "Regresión Lineal con Interacciones":
    modelo_lineal_mejor_rm = modelo_lineal_interacciones_rm
    coef_lineal_rm = modelo_lineal_mejor_rm.params.reset_index()
    coef_lineal_rm.columns = ["variable", "coeficiente"]
    coef_lineal_rm["p_value"] = modelo_lineal_mejor_rm.pvalues.values

elif mejor_lineal_rm == "OLS refit post-LASSO":
    modelo_lineal_mejor_rm = modelo_ols_post_lasso_rm
    coef_lineal_rm = modelo_lineal_mejor_rm.params.reset_index()
    coef_lineal_rm.columns = ["variable", "coeficiente"]
    coef_lineal_rm["p_value"] = modelo_lineal_mejor_rm.pvalues.values

elif mejor_lineal_rm == "LASSO":
    coef_lineal_rm = variables_seleccionadas_rm.copy()
    coef_lineal_rm = coef_lineal_rm.rename(
        columns={"coeficiente_lasso": "coeficiente"}
    )
    coef_lineal_rm["p_value"] = np.nan

coef_lineal_rm["abs_coeficiente"] = coef_lineal_rm["coeficiente"].abs()

coef_lineal_rm = coef_lineal_rm.sort_values(
    "abs_coeficiente",
    ascending=False
)

display(coef_lineal_rm.head(30))

## Extensión del modelo a 1 mes de datos! 